# Processing

In [ ]:
from pathlib import Path
import pandas as pd

# --------------------
# PATH
# --------------------
input_path = Path("/Users/gsw512/Documents/E1.2_MGK/females/Pavlovian")

# --------------------
# INFORMATION
# --------------------
fiber_map = {
    "20": {"VS": "G0", "PFC": "G2"},
    "16": {"VS": "G0", "PFC": "G2"},
    "18": {"VS": "G14", "PFC": "G6"},
    "30": {"VS": "G0", "PFC": "G2"},
    "31": {"VS": "G14", "PFC": "G6"},
    "19": {"VS": "G0", "PFC": "G2"},
    "42": {"VS": "G0", "PFC": "G2"},
    "54": {"VS": "G14", "PFC": "G6"},
    "64": {"VS": "G0", "PFC": "G2"},
    "65": {"VS": "G4", "PFC": "G6"},
}

genotype_map = {
    "20": "WT",
    "16": "WT",
    "18": "WT",
    "42": "WT",
    "54": "WT",
    "30": "KI",
    "31": "KI",
    "19": "KI",
    "64": "KI",
    "65": "KI",
}

L470 = "2"  # Dopamine/green signal
L415 = "1"  # Isosbestic/control signal

session_order = ["Habituation", "Session_1_ZT6", "Session_2_ZT11"]

# --------------------
# FUNCTIONS
# --------------------
def parse_exp_folder(folder_name):
    parts = folder_name.split("_")
    return {
        "year": parts[0],
        "month": parts[1],
        "day": parts[2],
        "animal_A": parts[3],
        "animal_B": parts[4],
        "experiment": parts[5],
        "sex": parts[6],
    }

def collect_session_files(session_path):
    return {
        "behavior": list(session_path.glob("CI*.csv")),
        "ttl": list(session_path.glob("DigInput*.csv")),
        "photometry": list(session_path.glob("photo*.csv")),
    }

def get_animal_id_from_behavior_csv(csv_path, header_rows=18):
    df_header = pd.read_csv(csv_path, header=None, nrows=header_rows, usecols=[0,1], engine="python")
    matches = df_header[df_header[0].astype(str).str.contains("Animal ID", na=False)]
    if matches.empty:
        raise ValueError(f"Animal ID not found in {csv_path.name}")
    return str(matches.iloc[0, 1])

def get_ttl_for_animal(ttl_files, animal_id, animal_A, animal_B):
    for ttl_file in ttl_files:
        name_lower = ttl_file.name.lower()
        if animal_id == animal_A and "_a" in name_lower:
            return ttl_file
        elif animal_id == animal_B and "_b" in name_lower:
            return ttl_file
    return None

def get_session_start(ttl_files):
    if not ttl_files:
        raise ValueError("No TTL files provided")
    ttl_df = pd.read_csv(ttl_files[0])
    return ttl_df["SystemTimestamp"].iloc[0]

def extract_photometry_signals(photo_csv, fiber_assignment, time_offset=0.0):
    """
    Extract 470/415 signals for each fiber, keep separate time columns for each LED,
    and interpolate NaNs to get continuous traces.
    """
    df = pd.read_csv(photo_csv)
    df["LedState"] = df["LedState"].astype(str)

    signals = {}

    for region, g_channel in fiber_assignment.items():
        # 470 signal & its time
        mask_470 = df["LedState"] == L470
        sig_470 = df[g_channel].where(mask_470).reset_index(drop=True)
        time_470 = df["SystemTimestamp"].where(mask_470).reset_index(drop=True) - time_offset

        # Interpolate NaNs for 470
        sig_470_interp = sig_470.interpolate(method="linear", limit_direction="both")
        time_470_interp = time_470.interpolate(method="linear", limit_direction="both")

        # 415 signal & its time
        mask_415 = df["LedState"] == L415
        sig_415 = df[g_channel].where(mask_415).reset_index(drop=True)
        time_415 = df["SystemTimestamp"].where(mask_415).reset_index(drop=True) - time_offset

        # Interpolate NaNs for 415
        sig_415_interp = sig_415.interpolate(method="linear", limit_direction="both")
        time_415_interp = time_415.interpolate(method="linear", limit_direction="both")

        signals[region] = {
            "470": sig_470_interp,
            "time_470": time_470_interp,
            "415": sig_415_interp,
            "time_415": time_415_interp
        }

    return signals

def extract_behavior_events(behavior_csv, session_start=0.0):
    df = pd.read_csv(behavior_csv, skiprows=18)
    df.columns = df.columns.str.strip()
    required_cols = ["Evnt_Time", "Item_Name", "Arg1_Value"]
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"Behavior CSV {behavior_csv.name} missing required column: {col}")
    return df[["Evnt_Time", "Item_Name", "Arg1_Value"]]

# --------------------
# PROCESSING
# --------------------
subfolders = ["Habituation", "Session_1_ZT6", "Session_2_ZT11"]

for exp_folder in input_path.iterdir():
    if not exp_folder.is_dir():
        continue

    metadata = parse_exp_folder(exp_folder.name)
    print("\n==============================")
    print("Processing experiment:", metadata)

    for sub_name in subfolders:
        sub_path = exp_folder / sub_name
        if not sub_path.exists():
            print(f"\n⚠ Missing {sub_name}")
            continue

        session_files = collect_session_files(sub_path)
        session_animals = {}

        for beh_file in session_files["behavior"]:
            try:
                animal_id = get_animal_id_from_behavior_csv(beh_file)
                session_animals[animal_id] = {
                    "behavior_file": beh_file,
                    "ttl_files": session_files["ttl"],
                    "photometry_file": session_files["photometry"][0] if session_files["photometry"] else None,
                    "fibers": fiber_map.get(animal_id, {}),
                    "genotype": genotype_map.get(animal_id, "Unknown"),
                    "session_start": None,
                }
            except Exception as e:
                print(f"  ❌ {beh_file.name}: {e}")

        # Assign TTL & extract signals/behavior
        for animal_id, info in session_animals.items():
            if info["photometry_file"] is None or not info["fibers"]:
                continue

            ttl_file = get_ttl_for_animal(info["ttl_files"], animal_id, metadata["animal_A"], metadata["animal_B"])
            info["session_start"] = get_session_start([ttl_file]) if ttl_file else 0.0

            info["signals"] = extract_photometry_signals(info["photometry_file"], info["fibers"], info["session_start"])
            info["events"] = extract_behavior_events(info["behavior_file"], info["session_start"])

# Preview

In [ ]:
# --------------------
# PREVIEW STORED DATA (formatted for readability)
# --------------------

for exp_folder in input_path.iterdir():
    if not exp_folder.is_dir():
        continue

    metadata = parse_exp_folder(exp_folder.name)
    print("\n" + "="*50)
    print(f"EXPERIMENT: {metadata['experiment']} ({metadata['year']}-{metadata['month']}-{metadata['day']})")
    print(f"Animals: {metadata['animal_A']}, {metadata['animal_B']} | Sex: {metadata['sex']}")
    print("="*50)

    for sub_name in subfolders:
        sub_path = exp_folder / sub_name
        if not sub_path.exists():
            continue

        session_files = collect_session_files(sub_path)
        session_animals = {}

        # Build session_animals
        for beh_file in session_files["behavior"]:
            try:
                animal_id = get_animal_id_from_behavior_csv(beh_file)
                session_animals[animal_id] = {
                    "behavior_file": beh_file,
                    "ttl_files": session_files["ttl"],
                    "photometry_file": session_files["photometry"][0] if session_files["photometry"] else None,
                    "fibers": fiber_map.get(animal_id, {}),
                    "genotype": genotype_map.get(animal_id, "Unknown"),
                    "session_start": None,
                    "signals": None,
                    "events": None
                }
            except Exception as e:
                print(f"  ❌ {beh_file.name}: {e}")

        # Extract TTL-aligned signals and events
        for animal_id, info in session_animals.items():
            if info["photometry_file"] is None or not info["fibers"]:
                continue

            ttl_file = get_ttl_for_animal(info["ttl_files"], animal_id, metadata["animal_A"], metadata["animal_B"])
            info["session_start"] = get_session_start([ttl_file]) if ttl_file else 0.0

            info["signals"] = extract_photometry_signals(info["photometry_file"], info["fibers"], info["session_start"])
            info["events"] = extract_behavior_events(info["behavior_file"], info["session_start"])

        # ----------- Preview info for each animal -----------
        for animal_id, info in session_animals.items():
            print("\n" + "-"*40)
            print(f"Animal ID: {animal_id} | Genotype: {info['genotype']} | Session: {sub_name}")
            print("-"*40)
            print(f"Photometry file: {info['photometry_file'].name if info['photometry_file'] else 'None'}")
            print(f"Behavior file: {info['behavior_file'].name if info['behavior_file'] else 'None'}")
            print(f"Session start (TTL): {info['session_start']}\n")

            # Behavioral events preview
            if info["events"] is not None:
                print(">>> First 5 behavioral events:")
                print(info["events"].head().to_string(index=False))
                print()

            # Photometry signals preview
            if info["signals"] is not None:
                for region, signals in info["signals"].items():
                    print(f"--- Region: {region} ---")
                    print(f"Time 470 (s) preview: {signals['time_470'].head().tolist()}")
                    print(f"470 signal preview: {signals['470'].head().tolist()}")
                    print(f"Time 415 (s) preview: {signals['time_415'].head().tolist()}")
                    print(f"415 signal preview: {signals['415'].head().tolist()}\n")

    print("\n" + "="*50 + "\n\n")

# Processsing photometry data

In [ ]:
# ----------- CHOP -----------

for exp_folder in input_path.iterdir():
    if not exp_folder.is_dir():
        continue

    metadata = parse_exp_folder(exp_folder.name)
    print("\n" + "="*50)
    print(f"EXPERIMENT: {metadata['experiment']} ({metadata['year']}-{metadata['month']}-{metadata['day']})")
    print(f"Animals: {metadata['animal_A']}, {metadata['animal_B']} | Sex: {metadata['sex']}")
    print("="*50)

    for sub_name in subfolders:
        # Skip habituation sessions
        if "habituation" in sub_name.lower():
            print(f"Skipping habituation session: {sub_name}")
            continue

        sub_path = exp_folder / sub_name
        if not sub_path.exists():
            continue

        session_files = collect_session_files(sub_path)
        session_animals = {}

        # Build session_animals
        for beh_file in session_files["behavior"]:
            try:
                animal_id = get_animal_id_from_behavior_csv(beh_file)
                session_animals[animal_id] = {
                    "behavior_file": beh_file,
                    "ttl_files": session_files["ttl"],
                    "photometry_file": session_files["photometry"][0] if session_files["photometry"] else None,
                    "fibers": fiber_map.get(animal_id, {}),
                    "genotype": genotype_map.get(animal_id, "Unknown"),
                    "session_start": None,
                    "signals": None,
                    "events": None
                }
            except Exception as e:
                print(f"  ❌ {beh_file.name}: {e}")

        # Extract TTL-aligned signals and events
        for animal_id, info in session_animals.items():
            if info["photometry_file"] is None or not info["fibers"]:
                continue

            ttl_file = get_ttl_for_animal(info["ttl_files"], animal_id, metadata["animal_A"], metadata["animal_B"])
            info["session_start"] = get_session_start([ttl_file]) if ttl_file else 0.0

            info["signals"] = extract_photometry_signals(info["photometry_file"], info["fibers"], info["session_start"])
            info["events"] = extract_behavior_events(info["behavior_file"], info["session_start"])

            # ---- Chop to -900 ----
            for region, signals in info["signals"].items():
                mask_470 = signals["time_470"] >= -900
                mask_415 = signals["time_415"] >= -900

                signals["time_470"] = signals["time_470"][mask_470]
                signals["470"] = signals["470"][mask_470]

                signals["time_415"] = signals["time_415"][mask_415]
                signals["415"] = signals["415"][mask_415]

            if info["events"] is not None:
                info["events"] = info["events"][info["events"]["Evnt_Time"] >= -900]
            
            if info["events"] is not None and not info["events"].empty:
                # Find maximum behavioral event time
                max_beh_time = info["events"]["Evnt_Time"].max()

                # Chop photometry signals at max behavioral event time
                for region, signals in info["signals"].items():
                    mask_470 = (signals["time_470"] >= -900) & (signals["time_470"] <= max_beh_time)
                    mask_415 = (signals["time_415"] >= -900) & (signals["time_415"] <= max_beh_time)

                    signals["time_470"] = signals["time_470"][mask_470]
                    signals["470"] = signals["470"][mask_470]

                    signals["time_415"] = signals["time_415"][mask_415]
                    signals["415"] = signals["415"][mask_415]

        # ----------- Preview info for each animal -----------
        for animal_id, info in session_animals.items():
            print("\n" + "-"*40)
            print(f"Animal ID: {animal_id} | Genotype: {info['genotype']} | Session: {sub_name}")
            print("-"*40)
            print(f"Session start (TTL): {info['session_start']}\n")

            # Behavioral events preview
            if info["events"] is not None and not info["events"].empty:
                print(">>> First 5 behavioral events (post-chop):")
                print(info["events"].head().to_string(index=False))
                print()

                print(">>> Last 5 behavioral events (post-chop):")
                print(info["events"].tail().to_string(index=False))
                print()

            # Photometry signals preview
            if info["signals"] is not None:
                for region, signals in info["signals"].items():
                    print(f"--- Region: {region} ---")
                    
                    # Show first 5 samples (already there)
                    print(f"Time 470 (start) preview: {signals['time_470'].head().tolist()}")
                    print(f"470 signal (start) preview: {signals['470'].head().tolist()}")
                    print(f"Time 415 (start) preview: {signals['time_415'].head().tolist()}")
                    print(f"415 signal (start) preview: {signals['415'].head().tolist()}")
                    
                    # Show last 5 samples
                    print(f"Time 470 (end) preview: {signals['time_470'].tail().tolist()}")
                    print(f"470 signal (end) preview: {signals['470'].tail().tolist()}")
                    print(f"Time 415 (end) preview: {signals['time_415'].tail().tolist()}")
                    print(f"415 signal (end) preview: {signals['415'].tail().tolist()}\n")

    print("\n" + "="*50 + "\n\n")

In [ ]:
import numpy as np

# ==========================================
# REWARD + REWARD COLLECT SUMMARY
# ==========================================

for exp_folder in input_path.iterdir():
    if not exp_folder.is_dir():
        continue

    metadata = parse_exp_folder(exp_folder.name)
    print("\n" + "=" * 60)
    print(f"EXPERIMENT: {metadata['experiment']} ({metadata['year']}-{metadata['month']}-{metadata['day']})")
    print(f"Animals: {metadata['animal_A']}, {metadata['animal_B']} | Sex: {metadata['sex']}")
    print("=" * 60)

    for sub_name in subfolders:
        if "habituation" in sub_name.lower():
            continue

        sub_path = exp_folder / sub_name
        if not sub_path.exists():
            continue

        print(f"\n--- SESSION: {sub_name} ---")

        session_files = collect_session_files(sub_path)

        for beh_file in session_files["behavior"]:
            try:
                animal_id = get_animal_id_from_behavior_csv(beh_file)

                # TTL alignment
                ttl_file = get_ttl_for_animal(
                    session_files["ttl"],
                    animal_id,
                    metadata["animal_A"],
                    metadata["animal_B"]
                )

                session_start = get_session_start([ttl_file]) if ttl_file else 0.0

                # Load behavior
                events = extract_behavior_events(beh_file, session_start)

                if events is None or events.empty:
                    print(f"\nAnimal {animal_id}: No behavioral data")
                    continue

                # Chop
                events = events[events["Evnt_Time"] >= -900]

                # Reward events
                reward_events = events[events["Item_Name"] == "reward"]

                print("\n" + "-" * 40)
                print(f"Animal ID: {animal_id}")
                print("-" * 40)

                if reward_events.empty:
                    print("No reward events found.")
                    continue

                print(f"Number of rewards: {len(reward_events)}")

                # -------- Find reward collect times --------
                reward_collect_times = []
                reward_latencies = []

                for r_time in reward_events["Evnt_Time"].values:
                    post_reward = events[
                        (events["Evnt_Time"] > r_time) &
                        (events["Item_Name"] == "ITI_head_entry")
                    ]

                    if post_reward.empty:
                        reward_collect_times.append(np.nan)
                        reward_latencies.append(np.nan)
                    else:
                        collect_time = post_reward.iloc[0]["Evnt_Time"]
                        reward_collect_times.append(collect_time)
                        reward_latencies.append(collect_time - r_time)

                # -------- Print summary --------
                summary_df = reward_events[["Evnt_Time"]].copy()
                summary_df.rename(columns={"Evnt_Time": "Reward_Time"}, inplace=True)
                summary_df["Reward_Collect_Time"] = reward_collect_times
                summary_df["Collect_Latency"] = reward_latencies

                print(summary_df.to_string(index=False))

            except Exception as e:
                print(f"❌ Error processing {beh_file.name}: {e}")

    print("\n" + "=" * 60 + "\n")

# Plotting raw traces

In [ ]:
import matplotlib.pyplot as plt

# ----------- COLLECT CHOPPED PHOTOMETRY WITH REWARDS + COLLECT -----------

all_plots = []   # <--- global container

for exp_folder in input_path.iterdir():
    if not exp_folder.is_dir():
        continue

    metadata = parse_exp_folder(exp_folder.name)

    for sub_name in subfolders:
        if "habituation" in sub_name.lower():
            continue

        sub_path = exp_folder / sub_name
        if not sub_path.exists():
            continue

        session_files = collect_session_files(sub_path)

        for beh_file in session_files["behavior"]:
            try:
                animal_id = get_animal_id_from_behavior_csv(beh_file)
            except Exception as e:
                print(f"❌ {beh_file.name}: {e}")
                continue

            photometry_file = (
                session_files["photometry"][0]
                if session_files["photometry"]
                else None
            )

            fibers = fiber_map.get(animal_id, {})
            genotype = genotype_map.get(animal_id, "Unknown")

            if photometry_file is None or not fibers:
                continue

            ttl_file = get_ttl_for_animal(
                session_files["ttl"],
                animal_id,
                metadata["animal_A"],
                metadata["animal_B"]
            )

            session_start = get_session_start([ttl_file]) if ttl_file else 0.0

            signals = extract_photometry_signals(
                photometry_file,
                fibers,
                session_start
            )

            events = extract_behavior_events(
                beh_file,
                session_start
            )

            if events is None or events.empty:
                continue

            # Filter rewards
            reward_events = events[events["Item_Name"] == "reward"]
            if reward_events.empty:
                continue

            max_reward_time = reward_events["Evnt_Time"].max()

            # ---- Chop signals ----
            for region, sig in signals.items():
                for led in ["470", "415"]:
                    tkey = f"time_{led}"
                    mask = (sig[tkey] >= -900) & (sig[tkey] <= max_reward_time)
                    sig[tkey] = sig[tkey][mask]
                    sig[led] = sig[led][mask]

                # ---- Extract reward-collect times ----
                collect_times = []
                for r_time in reward_events["Evnt_Time"].values:
                    post_reward = events[
                        (events["Evnt_Time"] > r_time) &
                        (events["Item_Name"] == "ITI_head_entry")
                    ]
                    if not post_reward.empty:
                        collect_times.append(post_reward.iloc[0]["Evnt_Time"])
                    else:
                        collect_times.append(None)

                # ---- Store ONE plot entry per region ----
                all_plots.append({
                    "animal_id"     : animal_id,
                    "genotype"      : genotype,
                    "experiment"    : metadata["experiment"],
                    "session"       : sub_name,
                    "region"        : region,
                    "signals"       : sig,
                    "reward_times"  : reward_events["Evnt_Time"].values,
                    "collect_times" : collect_times
                })

In [ ]:
# ----------- GLOBAL SORT (ACROSS ALL SESSIONS) -----------

def animal_sort_key(entry):
    aid = entry["animal_id"]
    return int(aid) if str(aid).isdigit() else str(aid)

all_plots_sorted = sorted(all_plots, key=animal_sort_key)

In [ ]:
# ----------- PLOT (SORTED ACROSS ALL SESSIONS) -----------

for entry in all_plots_sorted:
    sig = entry["signals"]

    plt.figure(figsize=(24, 5))

    # Photometry
    plt.plot(
        sig["time_470"],
        sig["470"],
        label="470 nm",
        color="green",
        lw=0.8
    )
    plt.plot(
        sig["time_415"],
        sig["415"],
        label="415 nm",
        color="gray",
        alpha=0.7,
        lw=0.8
    )

    # Reward delivery (red)
    for i, t in enumerate(entry["reward_times"]):
        plt.axvline(
            t,
            color="red",
            linestyle="--",
            alpha=0.6,
            label="Reward" if i == 0 else None
        )

    # Reward collect (purple)
    for i, t in enumerate(entry["collect_times"]):
        if t is None:
            continue
        plt.axvline(
            t,
            color="purple",
            linestyle=":",
            alpha=0.8,
            label="Reward collect" if i == 0 else None
        )

    plt.title(
        f"Animal {entry['animal_id']} | "
        f"Genotype {entry['genotype']} | "
        f"Region {entry['region']} | "
        f"Session {entry['session']} | "
        f"Exp {entry['experiment']}"
    )

    plt.xlabel("Time (s, chopped)")
    plt.ylabel("Fluorescence")
    plt.legend()
    plt.tight_layout()
    plt.show()

# Plotting normalized traces

## (470-415)/415

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ----------- COLLECT CHOPPED PHOTOMETRY WITH ΔF/F + REWARDS + COLLECT -----------

all_plots = []   # <--- global container

for exp_folder in input_path.iterdir():
    if not exp_folder.is_dir():
        continue

    metadata = parse_exp_folder(exp_folder.name)

    for sub_name in subfolders:
        if "habituation" in sub_name.lower():
            continue

        sub_path = exp_folder / sub_name
        if not sub_path.exists():
            continue

        session_files = collect_session_files(sub_path)

        for beh_file in session_files["behavior"]:
            try:
                animal_id = get_animal_id_from_behavior_csv(beh_file)
            except Exception as e:
                print(f"❌ {beh_file.name}: {e}")
                continue

            photometry_file = (
                session_files["photometry"][0]
                if session_files["photometry"]
                else None
            )

            fibers = fiber_map.get(animal_id, {})
            genotype = genotype_map.get(animal_id, "Unknown")

            if photometry_file is None or not fibers:
                continue

            ttl_file = get_ttl_for_animal(
                session_files["ttl"],
                animal_id,
                metadata["animal_A"],
                metadata["animal_B"]
            )

            session_start = get_session_start([ttl_file]) if ttl_file else 0.0

            signals = extract_photometry_signals(
                photometry_file,
                fibers,
                session_start
            )

            events = extract_behavior_events(
                beh_file,
                session_start
            )

            if events is None or events.empty:
                continue

            # Filter rewards
            reward_events = events[events["Item_Name"] == "reward"]
            if reward_events.empty:
                continue

            max_reward_time = reward_events["Evnt_Time"].max()

            # ---- Chop signals and compute dF/F ----
            for region, sig in signals.items():
                for led in ["470", "415"]:
                    tkey = f"time_{led}"
                    mask = (sig[tkey] >= -900) & (sig[tkey] <= max_reward_time)
                    sig[tkey] = sig[tkey][mask]
                    sig[led] = sig[led][mask]

                # Compute simple dF/F
                # Avoid division by zero
                sig["dF/F"] = np.where(sig["415"] != 0, (sig["470"] - sig["415"]) / sig["415"], np.nan)

                # ---- Extract reward-collect times ----
                collect_times = []
                for r_time in reward_events["Evnt_Time"].values:
                    post_reward = events[
                        (events["Evnt_Time"] > r_time) &
                        (events["Item_Name"] == "ITI_head_entry")
                    ]
                    if not post_reward.empty:
                        collect_times.append(post_reward.iloc[0]["Evnt_Time"])
                    else:
                        collect_times.append(None)

                # ---- Store ONE plot entry per region ----
                all_plots.append({
                    "animal_id"     : animal_id,
                    "genotype"      : genotype,
                    "experiment"    : metadata["experiment"],
                    "session"       : sub_name,
                    "region"        : region,
                    "signals"       : sig,
                    "reward_times"  : reward_events["Evnt_Time"].values,
                    "collect_times" : collect_times
                })

In [ ]:
# ----------- GLOBAL SORT (ACROSS ALL SESSIONS) -----------

def animal_sort_key(entry):
    aid = entry["animal_id"]
    return int(aid) if str(aid).isdigit() else str(aid)

all_plots_sorted = sorted(all_plots, key=animal_sort_key)

In [ ]:
# ----------- PLOT ΔF/F (SORTED ACROSS ALL SESSIONS) -----------

for entry in all_plots_sorted:
    sig = entry["signals"]

    plt.figure(figsize=(24, 5))

    # ΔF/F trace
    plt.plot(
        sig["time_470"],
        sig["dF/F"],
        label="ΔF/F",
        color="black",
        lw=0.8
    )

    # Reward delivery (red)
    for i, t in enumerate(entry["reward_times"]):
        plt.axvline(
            t,
            color="red",
            linestyle="--",
            alpha=0.6,
            label="Reward" if i == 0 else None
        )

    # Reward collect (purple)
    for i, t in enumerate(entry["collect_times"]):
        if t is None:
            continue
        plt.axvline(
            t,
            color="purple",
            linestyle=":",
            alpha=0.8,
            label="Reward collect" if i == 0 else None
        )

    plt.title(
        f"Animal {entry['animal_id']} | "
        f"Genotype {entry['genotype']} | "
        f"Region {entry['region']} | "
        f"Session {entry['session']} | "
        f"Exp {entry['experiment']}"
    )

    plt.xlabel("Time (s, chopped)")
    plt.ylabel("ΔF/F")
    plt.legend()
    plt.tight_layout()
    plt.show()

## fitted 415

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ----------- COLLECT CHOPPED PHOTOMETRY WITH FITTED ΔF/F + REWARDS + COLLECT -----------

all_plots = []   # <--- global container

for exp_folder in input_path.iterdir():
    if not exp_folder.is_dir():
        continue

    metadata = parse_exp_folder(exp_folder.name)

    for sub_name in subfolders:
        if "habituation" in sub_name.lower():
            continue

        sub_path = exp_folder / sub_name
        if not sub_path.exists():
            continue

        session_files = collect_session_files(sub_path)

        for beh_file in session_files["behavior"]:
            try:
                animal_id = get_animal_id_from_behavior_csv(beh_file)
            except Exception as e:
                print(f"❌ {beh_file.name}: {e}")
                continue

            photometry_file = (
                session_files["photometry"][0]
                if session_files["photometry"]
                else None
            )

            fibers = fiber_map.get(animal_id, {})
            genotype = genotype_map.get(animal_id, "Unknown")

            if photometry_file is None or not fibers:
                continue

            ttl_file = get_ttl_for_animal(
                session_files["ttl"],
                animal_id,
                metadata["animal_A"],
                metadata["animal_B"]
            )

            session_start = get_session_start([ttl_file]) if ttl_file else 0.0

            signals = extract_photometry_signals(
                photometry_file,
                fibers,
                session_start
            )

            events = extract_behavior_events(
                beh_file,
                session_start
            )

            if events is None or events.empty:
                continue

            # Filter rewards
            reward_events = events[events["Item_Name"] == "reward"]
            if reward_events.empty:
                continue

            max_reward_time = reward_events["Evnt_Time"].max()

            # ---- Chop signals and compute fitted ΔF/F ----
            for region, sig in signals.items():

                # Chop both channels identically
                for led in ["470", "415"]:
                    tkey = f"time_{led}"
                    mask = (sig[tkey] >= -900) & (sig[tkey] <= max_reward_time)
                    sig[tkey] = sig[tkey][mask]
                    sig[led] = sig[led][mask]

                # Alias for clarity
                sig470 = sig["470"]
                ctrl415 = sig["415"]

                # Remove NaNs / infs before fitting
                valid = np.isfinite(sig470) & np.isfinite(ctrl415)

                if valid.sum() < 10:
                    # Not enough points to fit
                    sig["dF/F"] = np.full_like(sig470, np.nan)
                else:
                    # Linear fit: 415 -> 470
                    m, b = np.polyfit(ctrl415[valid], sig470[valid], 1)
                    fitted = m * ctrl415 + b

                    # Avoid division by zero
                    sig["dF/F"] = np.where(
                        fitted != 0,
                        (sig470 - fitted) / fitted * 100,   # percent ΔF/F
                        np.nan
                    )

                # ---- Extract reward-collect times ----
                collect_times = []
                for r_time in reward_events["Evnt_Time"].values:
                    post_reward = events[
                        (events["Evnt_Time"] > r_time) &
                        (events["Item_Name"] == "ITI_head_entry")
                    ]
                    if not post_reward.empty:
                        collect_times.append(post_reward.iloc[0]["Evnt_Time"])
                    else:
                        collect_times.append(None)

                # ---- Store ONE plot entry per region ----
                all_plots.append({
                    "animal_id"     : animal_id,
                    "genotype"      : genotype,
                    "experiment"    : metadata["experiment"],
                    "session"       : sub_name,
                    "region"        : region,
                    "signals"       : sig,
                    "reward_times"  : reward_events["Evnt_Time"].values,
                    "collect_times" : collect_times
                })

In [ ]:
# ----------- GLOBAL SORT (ACROSS ALL SESSIONS) -----------

def animal_sort_key(entry):
    aid = entry["animal_id"]
    return int(aid) if str(aid).isdigit() else str(aid)

all_plots_sorted = sorted(all_plots, key=animal_sort_key)

In [ ]:
# ----------- PLOT FITTED ΔF/F (SORTED ACROSS ALL SESSIONS) -----------

for entry in all_plots_sorted:
    sig = entry["signals"]

    plt.figure(figsize=(24, 5))

    plt.plot(
        sig["time_470"],
        sig["dF/F"],
        label="ΔF/F (fitted 415)",
        color="black",
        lw=0.8
    )

    # Reward delivery (red)
    for i, t in enumerate(entry["reward_times"]):
        plt.axvline(
            t,
            color="red",
            linestyle="--",
            alpha=0.6,
            label="Reward" if i == 0 else None
        )

    # Reward collect (purple)
    for i, t in enumerate(entry["collect_times"]):
        if t is None:
            continue
        plt.axvline(
            t,
            color="purple",
            linestyle=":",
            alpha=0.8,
            label="Reward collect" if i == 0 else None
        )

    plt.title(
        f"Animal {entry['animal_id']} | "
        f"Genotype {entry['genotype']} | "
        f"Region {entry['region']} | "
        f"Session {entry['session']} | "
        f"Exp {entry['experiment']}"
    )

    plt.xlabel("Time (s, chopped)")
    plt.ylabel("ΔF/F (%)")
    plt.legend()
    plt.tight_layout()
    plt.show()

## fitted 415 - baseline

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# =========================================================
# COLLECT CHOPPED PHOTOMETRY WITH BASELINE-FITTED ΔF/F
# =========================================================

all_plots = []

BASELINE_START = -900
BASELINE_END   = 0

for exp_folder in input_path.iterdir():
    if not exp_folder.is_dir():
        continue

    metadata = parse_exp_folder(exp_folder.name)

    for sub_name in subfolders:
        if "habituation" in sub_name.lower():
            continue

        sub_path = exp_folder / sub_name
        if not sub_path.exists():
            continue

        session_files = collect_session_files(sub_path)

        for beh_file in session_files["behavior"]:
            try:
                animal_id = get_animal_id_from_behavior_csv(beh_file)
            except Exception as e:
                print(f"❌ {beh_file.name}: {e}")
                continue

            photometry_file = (
                session_files["photometry"][0]
                if session_files["photometry"]
                else None
            )

            fibers = fiber_map.get(animal_id, {})
            genotype = genotype_map.get(animal_id, "Unknown")

            if photometry_file is None or not fibers:
                continue

            ttl_file = get_ttl_for_animal(
                session_files["ttl"],
                animal_id,
                metadata["animal_A"],
                metadata["animal_B"]
            )

            session_start = get_session_start([ttl_file]) if ttl_file else 0.0

            signals = extract_photometry_signals(
                photometry_file,
                fibers,
                session_start
            )

            events = extract_behavior_events(
                beh_file,
                session_start
            )

            if events is None or events.empty:
                continue

            # Reward events
            reward_events = events[events["Item_Name"] == "reward"]
            if reward_events.empty:
                continue

            max_reward_time = reward_events["Evnt_Time"].max()

            # -------------------------------------------------
            # Process each region
            # -------------------------------------------------
            for region, sig in signals.items():

                # Chop both channels identically
                for led in ["470", "415"]:
                    tkey = f"time_{led}"
                    mask = (sig[tkey] >= BASELINE_START) & (sig[tkey] <= max_reward_time)
                    sig[tkey] = sig[tkey][mask]
                    sig[led] = sig[led][mask]

                time = sig["time_470"]
                sig470 = sig["470"]
                ctrl415 = sig["415"]

                # -------------------------------------------------
                # Baseline-restricted linear fit
                # -------------------------------------------------
                baseline_mask = (time >= BASELINE_START) & (time <= BASELINE_END)
                valid_mask = (
                    baseline_mask &
                    np.isfinite(sig470) &
                    np.isfinite(ctrl415)
                )

                if valid_mask.sum() < 10:
                    sig["dF/F"] = np.full_like(sig470, np.nan)
                else:
                    # Fit 415 -> 470 using baseline only
                    m, b = np.polyfit(ctrl415[valid_mask], sig470[valid_mask], 1)

                    # Predict fitted baseline for all timepoints
                    fitted = m * ctrl415 + b

                    # ΔF/F (%)
                    sig["dF/F"] = np.where(
                        fitted != 0,
                        (sig470 - fitted) / fitted * 100,
                        np.nan
                    )

                # -------------------------------------------------
                # Reward collect times
                # -------------------------------------------------
                collect_times = []
                for r_time in reward_events["Evnt_Time"].values:
                    post_reward = events[
                        (events["Evnt_Time"] > r_time) &
                        (events["Item_Name"] == "ITI_head_entry")
                    ]
                    if not post_reward.empty:
                        collect_times.append(post_reward.iloc[0]["Evnt_Time"])
                    else:
                        collect_times.append(None)

                # -------------------------------------------------
                # Store plot entry
                # -------------------------------------------------
                all_plots.append({
                    "animal_id"     : animal_id,
                    "genotype"      : genotype,
                    "experiment"    : metadata["experiment"],
                    "session"       : sub_name,
                    "region"        : region,
                    "signals"       : sig,
                    "reward_times"  : reward_events["Evnt_Time"].values,
                    "collect_times" : collect_times
                })

# =========================================================
# GLOBAL SORT (ACROSS ALL SESSIONS)
# =========================================================

def animal_sort_key(entry):
    aid = entry["animal_id"]
    return int(aid) if str(aid).isdigit() else str(aid)

all_plots_sorted = sorted(all_plots, key=animal_sort_key)

# =========================================================
# PLOT BASELINE-FITTED ΔF/F
# =========================================================

for entry in all_plots_sorted:
    sig = entry["signals"]

    plt.figure(figsize=(24, 5))

    plt.plot(
        sig["time_470"],
        sig["dF/F"],
        color="black",
        lw=0.8,
        label="ΔF/F (baseline-fitted 415)"
    )

    # Reward delivery
    for i, t in enumerate(entry["reward_times"]):
        plt.axvline(
            t,
            color="red",
            linestyle="--",
            alpha=0.6,
            label="Reward" if i == 0 else None
        )

    # Reward collection
    for i, t in enumerate(entry["collect_times"]):
        if t is None:
            continue
        plt.axvline(
            t,
            color="purple",
            linestyle=":",
            alpha=0.8,
            label="Reward collect" if i == 0 else None
        )

    plt.title(
        f"Animal {entry['animal_id']} | "
        f"Genotype {entry['genotype']} | "
        f"Region {entry['region']} | "
        f"Session {entry['session']} | "
        f"Exp {entry['experiment']}"
    )

    plt.xlabel("Time (s)")
    plt.ylabel("ΔF/F (%)")
    plt.legend()
    plt.tight_layout()
    plt.show()

## fitted 415 - baseline - high pass filter

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt

# =========================================================
# PARAMETERS
# =========================================================

BASELINE_START = -900
BASELINE_END   = 0

HP_CUTOFF_HZ = 0.01   # set to None to disable HP filtering
HP_ORDER     = 2

# =========================================================
# HELPERS
# =========================================================

def highpass_filter(signal, fs, cutoff, order=2):
    nyq = 0.5 * fs
    norm_cutoff = cutoff / nyq
    b, a = butter(order, norm_cutoff, btype="highpass")
    return filtfilt(b, a, signal)

# =========================================================
# COLLECT + NORMALIZE
# =========================================================

all_plots = []

for exp_folder in input_path.iterdir():
    if not exp_folder.is_dir():
        continue

    metadata = parse_exp_folder(exp_folder.name)

    for sub_name in subfolders:
        if "habituation" in sub_name.lower():
            continue

        sub_path = exp_folder / sub_name
        if not sub_path.exists():
            continue

        session_files = collect_session_files(sub_path)

        for beh_file in session_files["behavior"]:
            try:
                animal_id = get_animal_id_from_behavior_csv(beh_file)
            except Exception as e:
                print(f"❌ {beh_file.name}: {e}")
                continue

            photometry_file = (
                session_files["photometry"][0]
                if session_files["photometry"]
                else None
            )

            fibers = fiber_map.get(animal_id, {})
            genotype = genotype_map.get(animal_id, "Unknown")

            if photometry_file is None or not fibers:
                continue

            ttl_file = get_ttl_for_animal(
                session_files["ttl"],
                animal_id,
                metadata["animal_A"],
                metadata["animal_B"]
            )

            session_start = get_session_start([ttl_file]) if ttl_file else 0.0

            signals = extract_photometry_signals(
                photometry_file,
                fibers,
                session_start
            )

            events = extract_behavior_events(
                beh_file,
                session_start
            )

            if events is None or events.empty:
                continue

            reward_events = events[events["Item_Name"] == "reward"]
            if reward_events.empty:
                continue

            max_reward_time = reward_events["Evnt_Time"].max()

            # -------------------------------------------------
            # Process each region
            # -------------------------------------------------
            for region, sig in signals.items():

                # Chop signals
                for led in ["470", "415"]:
                    tkey = f"time_{led}"
                    mask = (sig[tkey] >= BASELINE_START) & (sig[tkey] <= max_reward_time)
                    sig[tkey] = sig[tkey][mask]
                    sig[led] = sig[led][mask]

                time = sig["time_470"]
                sig470 = sig["470"]
                ctrl415 = sig["415"]

                # -------------------------------------------------
                # Sampling rate from time
                # -------------------------------------------------
                dt = np.median(np.diff(time))
                fs = 1.0 / dt

                # -------------------------------------------------
                # Baseline-restricted linear fit (RAW signals)
                # -------------------------------------------------
                baseline_mask = (time >= BASELINE_START) & (time <= BASELINE_END)
                valid_mask = (
                    baseline_mask &
                    np.isfinite(sig470) &
                    np.isfinite(ctrl415)
                )

                if valid_mask.sum() < 10:
                    sig["dF/F"] = np.full_like(sig470, np.nan)
                    continue

                m, b = np.polyfit(
                    ctrl415[valid_mask],
                    sig470[valid_mask],
                    1
                )

                # Raw fitted baseline
                fitted_raw = m * ctrl415 + b

                # Safeguard: reject bad fits
                if np.any(fitted_raw <= 0):
                    sig["dF/F"] = np.full_like(sig470, np.nan)
                    continue

                # -------------------------------------------------
                # Raw ΔF/F (%)
                # -------------------------------------------------
                dff_raw = (sig470 - fitted_raw) / fitted_raw * 100

                # -------------------------------------------------
                # Optional high-pass filter ΔF/F
                # -------------------------------------------------
                if HP_CUTOFF_HZ is not None:
                    try:
                        dff = highpass_filter(dff_raw, fs, HP_CUTOFF_HZ, HP_ORDER)
                    except ValueError:
                        dff = np.full_like(dff_raw, np.nan)
                else:
                    dff = dff_raw

                sig["dF/F"] = dff

                # -------------------------------------------------
                # Reward collect times
                # -------------------------------------------------
                collect_times = []
                for r_time in reward_events["Evnt_Time"].values:
                    post_reward = events[
                        (events["Evnt_Time"] > r_time) &
                        (events["Item_Name"] == "ITI_head_entry")
                    ]
                    if not post_reward.empty:
                        collect_times.append(post_reward.iloc[0]["Evnt_Time"])
                    else:
                        collect_times.append(None)

                # -------------------------------------------------
                # Store
                # -------------------------------------------------
                all_plots.append({
                    "animal_id"     : animal_id,
                    "genotype"      : genotype,
                    "experiment"    : metadata["experiment"],
                    "session"       : sub_name,
                    "region"        : region,
                    "signals"       : sig,
                    "reward_times"  : reward_events["Evnt_Time"].values,
                    "collect_times" : collect_times
                })

# =========================================================
# SORT
# =========================================================

def animal_sort_key(entry):
    aid = entry["animal_id"]
    return int(aid) if str(aid).isdigit() else str(aid)

all_plots_sorted = sorted(all_plots, key=animal_sort_key)

# =========================================================
# PLOT
# =========================================================

for entry in all_plots_sorted:
    sig = entry["signals"]

    plt.figure(figsize=(24, 5))

    plt.plot(
        sig["time_470"],
        sig["dF/F"],
        color="black",
        lw=0.8,
        label="ΔF/F (baseline-fit, safe)"
    )

    for i, t in enumerate(entry["reward_times"]):
        plt.axvline(
            t, color="red", linestyle="--",
            alpha=0.6, label="Reward" if i == 0 else None
        )

    for i, t in enumerate(entry["collect_times"]):
        if t is None:
            continue
        plt.axvline(
            t, color="purple", linestyle=":",
            alpha=0.8, label="Reward collect" if i == 0 else None
        )

    plt.title(
        f"Animal {entry['animal_id']} | "
        f"Genotype {entry['genotype']} | "
        f"Region {entry['region']} | "
        f"Session {entry['session']} | "
        f"Exp {entry['experiment']}"
    )

    plt.xlabel("Time (s)")
    plt.ylabel("ΔF/F (%)")
    plt.legend()
    plt.tight_layout()
    plt.show()
    

### Reward

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt

# =========================================================
# USER SETTINGS
# =========================================================
HP_CUTOFF_HZ = 0.03     # Hz (event-scale detrending)
HP_ORDER     = 3
WINDOW_SEC   = 10       # seconds before/after reward
BASELINE_START = -10    # start of baseline window
BASELINE_END   = -5     # end of baseline window

# =========================================================
# FILTER FUNCTION
# =========================================================
def highpass_filter(signal, cutoff, fs, order=3):
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    b, a = butter(order, normal_cutoff, btype="high")
    return filtfilt(b, a, signal)

# =========================================================
# PERI-REWARD PLOTS WITH BASELINE CORRECTION
# =========================================================
for entry in all_plots_sorted:   # <-- from your normalization pipeline
    sig = entry["signals"]

    time = sig["time_470"]
    dff  = sig["dF/F"]           # baseline-fitted ΔF/F (already safe)

    # -----------------------------------------------------
    # Sampling rate from time
    # -----------------------------------------------------
    dt = np.median(np.diff(time))
    fs = 1.0 / dt

    # -----------------------------------------------------
    # High-pass filter ΔF/F (event-scale)
    # -----------------------------------------------------
    try:
        dff_hp = highpass_filter(dff, HP_CUTOFF_HZ, fs, HP_ORDER)
    except ValueError:
        continue

    plt.figure(figsize=(14, 5))

    # -----------------------------------------------------
    # Plot each reward-aligned trace
    # -----------------------------------------------------
    for r_time in entry["reward_times"]:

        t_aligned = time - r_time
        mask = (t_aligned >= -WINDOW_SEC) & (t_aligned <= WINDOW_SEC)

        if mask.sum() < 5:
            continue

        # -------------------------------------------------
        # BASELINE CORRECTION
        # -------------------------------------------------
        baseline_mask = (t_aligned >= BASELINE_START) & (t_aligned <= BASELINE_END)
        if baseline_mask.sum() > 0:
            baseline_mean = np.nanmean(dff_hp[baseline_mask])
        else:
            baseline_mean = 0.0

        dff_corrected = dff_hp[mask] - baseline_mean

        plt.plot(
            t_aligned[mask],
            dff_corrected,
            color="black",
            lw=0.8,
            alpha=0.6
        )

    # Reference lines
    plt.axvline(0,  color="gray", linestyle="--", alpha=0.8)
    plt.axvline(BASELINE_END, color="gray", linestyle="--", alpha=0.5)

    plt.title(
        f"Animal {entry['animal_id']} | Genotype {entry['genotype']} | "
        f"Region {entry['region']} | Session {entry['session']} | "
        f"Exp {entry['experiment']}"
    )

    plt.xlabel("Time relative to reward (s)")
    plt.ylabel(f"ΔF/F (HP > {HP_CUTOFF_HZ} Hz, baseline-corrected)")
    plt.xlim(-WINDOW_SEC, WINDOW_SEC)
    plt.tight_layout()
    plt.show()

### Reward means

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt
from scipy.interpolate import interp1d

# =========================================================
# USER SETTINGS
# =========================================================
HP_CUTOFF_HZ    = 0.03       # Hz, high-pass filter for event-scale drift
HP_ORDER        = 3
WINDOW_SEC      = 10         # seconds before/after reward
RESAMPLE_POINTS = 800        # points for uniform alignment
BASELINE_START  = -10        # baseline window start
BASELINE_END    = -5         # baseline window end

# =========================================================
# FILTER FUNCTION
# =========================================================
def highpass_filter(signal, cutoff, fs, order=3):
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    b, a = butter(order, normal_cutoff, btype="high")
    return filtfilt(b, a, signal)

# =========================================================
# PERI-REWARD MEAN + SEM PLOTS WITH BASELINE CORRECTION
# =========================================================
for entry in all_plots_sorted:  # use baseline-fitted ΔF/F
    sig = entry["signals"]
    time = sig["time_470"]
    dff  = sig["dF/F"]  # baseline-fitted ΔF/F

    # -----------------------------------------------------
    # Compute sampling rate dynamically
    # -----------------------------------------------------
    dt = np.median(np.diff(time))
    fs = 1.0 / dt

    # -----------------------------------------------------
    # High-pass filter ΔF/F
    # -----------------------------------------------------
    try:
        dff_hp = highpass_filter(dff, HP_CUTOFF_HZ, fs, HP_ORDER)
    except ValueError:
        continue

    plt.figure(figsize=(14, 5))

    aligned_traces = []

    # Common resampled time vector
    t_resampled = np.linspace(-WINDOW_SEC, WINDOW_SEC, RESAMPLE_POINTS)

    # -----------------------------------------------------
    # Align each reward trial
    # -----------------------------------------------------
    for r_time in entry["reward_times"]:
        t_window = time - r_time
        mask = (t_window >= -WINDOW_SEC) & (t_window <= WINDOW_SEC)
        if not np.any(mask):
            continue

        # -------------------------------------------------
        # BASELINE CORRECTION per trial
        # -------------------------------------------------
        baseline_mask = (t_window >= BASELINE_START) & (t_window <= BASELINE_END)
        if baseline_mask.sum() > 0:
            baseline_mean = np.nanmean(dff_hp[baseline_mask])
        else:
            baseline_mean = 0.0

        dff_corrected = dff_hp[mask] - baseline_mean

        # -------------------------------------------------
        # Interpolate corrected trace onto uniform time vector
        # -------------------------------------------------
        interp_func = interp1d(
            t_window[mask],
            dff_corrected,
            kind='linear',
            bounds_error=False,
            fill_value=np.nan
        )
        aligned_traces.append(interp_func(t_resampled))

    # -----------------------------------------------------
    # Compute mean + SEM
    # -----------------------------------------------------
    if aligned_traces:
        aligned_array = np.vstack(aligned_traces)

        mean_trace = np.nanmean(aligned_array, axis=0)
        sem_trace  = np.nanstd(aligned_array, axis=0) / np.sqrt(np.sum(~np.isnan(aligned_array), axis=0))

        # Plot mean + SEM
        plt.plot(t_resampled, mean_trace, color="red", lw=2, label="Mean ΔF/F")
        plt.fill_between(t_resampled, mean_trace - sem_trace, mean_trace + sem_trace,
                         color="red", alpha=0.3, label="SEM")

        # Vertical lines: reward & marker
        plt.axvline(0,  color="gray", linestyle="--", alpha=0.8)  # reward
        plt.axvline(BASELINE_END, color="gray", linestyle="--", alpha=0.5)  # marker at end of baseline

    plt.title(
        f"Animal {entry['animal_id']} | Genotype {entry['genotype']} | "
        f"Region {entry['region']} | Session {entry['session']} | Exp {entry['experiment']}"
    )
    plt.xlabel("Time relative to reward (s)")
    plt.ylabel(f"ΔF/F (high-pass >{HP_CUTOFF_HZ} Hz, baseline-corrected)")
    plt.xlim(-WINDOW_SEC, WINDOW_SEC)
    plt.legend()
    plt.tight_layout()
    plt.show()

### Reward compiled means

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt
from collections import defaultdict

# =========================================================
# USER SETTINGS
# =========================================================
HP_CUTOFF_HZ    = 0.03     # Hz for event-scale high-pass filter
HP_ORDER        = 3
WINDOW_SEC      = 10       # seconds before/after reward
RESAMPLE_POINTS = 800      # points for uniform alignment
BASELINE_START  = -10      # baseline window start
BASELINE_END    = -5       # baseline window end

# =========================================================
# HIGH-PASS FILTER FUNCTION
# =========================================================
def highpass_filter(signal, cutoff, fs, order=3):
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    b, a = butter(order, normal_cutoff, btype="high")
    return filtfilt(b, a, signal)

# =========================================================
# GROUP DATA BY REGION × SESSION × GENOTYPE
# =========================================================
grouped_data = defaultdict(list)

for entry in all_plots_sorted:   # baseline-fitted ΔF/F
    sig = entry["signals"]
    time = sig["time_470"]
    dff  = sig["dF/F"]  # baseline-fitted ΔF/F

    # -----------------------------------------------------
    # Compute sampling rate dynamically
    # -----------------------------------------------------
    dt = np.median(np.diff(time))
    fs = 1.0 / dt

    # -----------------------------------------------------
    # High-pass filter ΔF/F
    # -----------------------------------------------------
    try:
        dff_hp = highpass_filter(dff, HP_CUTOFF_HZ, fs, HP_ORDER)
    except ValueError:
        continue

    t_resampled = np.linspace(-WINDOW_SEC, WINDOW_SEC, RESAMPLE_POINTS)
    aligned_traces = []

    # -----------------------------------------------------
    # Align reward trials with baseline correction
    # -----------------------------------------------------
    for r_time in entry["reward_times"]:
        t_window = time - r_time
        mask = (t_window >= -WINDOW_SEC) & (t_window <= WINDOW_SEC)
        if not np.any(mask):
            continue

        # Baseline correction per trial
        baseline_mask = (t_window >= BASELINE_START) & (t_window <= BASELINE_END)
        if baseline_mask.sum() > 0:
            baseline_mean = np.nanmean(dff_hp[baseline_mask])
        else:
            baseline_mean = 0.0

        corrected_trace = dff_hp[mask] - baseline_mean

        # Interpolate onto uniform time vector
        interp_func = np.interp(
            t_resampled,
            t_window[mask],
            corrected_trace,
            left=np.nan,
            right=np.nan
        )
        aligned_traces.append(interp_func)

    if aligned_traces:
        aligned_array = np.vstack(aligned_traces)
        mean_trace = np.nanmean(aligned_array, axis=0)
        sem_trace  = np.nanstd(aligned_array, axis=0) / np.sqrt(np.sum(~np.isnan(aligned_array), axis=0))

        key = (entry["region"], entry["session"], entry["genotype"])
        grouped_data[key].append({
            "animal_id": entry["animal_id"],
            "mean_trace": mean_trace,
            "sem_trace": sem_trace
        })

# =========================================================
# PLOT GROUPED MEAN TRACES
# =========================================================
for (region, session, genotype), animal_traces in grouped_data.items():
    plt.figure(figsize=(14, 5))
    t_resampled = np.linspace(-WINDOW_SEC, WINDOW_SEC, RESAMPLE_POINTS)

    # Plot each animal's mean trace
    for animal in animal_traces:
        plt.plot(t_resampled, animal["mean_trace"], lw=2, alpha=0.6, label=f"Animal {animal['animal_id']}")

    # Group mean ± SEM
    all_means = np.array([a["mean_trace"] for a in animal_traces])
    group_mean = np.nanmean(all_means, axis=0)
    group_sem  = np.nanstd(all_means, axis=0) / np.sqrt(len(animal_traces))

    plt.plot(t_resampled, group_mean, color="black", lw=3, label="Group mean")
    plt.fill_between(t_resampled, group_mean - group_sem, group_mean + group_sem,
                     color="gray", alpha=0.3, label="Group SEM")

    # Vertical reference lines
    plt.axvline(0, color="gray", linestyle="--", alpha=0.8)   # reward
    plt.axvline(BASELINE_END, color="gray", linestyle="--", alpha=0.5)  # baseline end marker

    plt.title(f"Region {region} | Session {session} | Genotype {genotype}")
    plt.xlabel("Time relative to reward (s)")
    plt.ylabel(f"ΔF/F (high-pass >{HP_CUTOFF_HZ} Hz, baseline-corrected)")
    plt.xlim(-WINDOW_SEC, WINDOW_SEC)
    plt.legend()
    plt.tight_layout()
    plt.show()

### Reward collect

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt

# =========================================================
# USER SETTINGS
# =========================================================
HP_CUTOFF_HZ   = 0.03   # Hz for event-scale high-pass
HP_ORDER       = 3
WINDOW_SEC     = 10     # seconds before/after reward collect
BASELINE_START = -10    # baseline window start
BASELINE_END   = -5     # baseline window end

# =========================================================
# HIGH-PASS FILTER FUNCTION
# =========================================================
def highpass_filter(signal, cutoff, fs, order=3):
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    b, a = butter(order, normal_cutoff, btype="high")
    return filtfilt(b, a, signal)

# =========================================================
# PERI-REWARD-COLLECT PLOTS WITH BASELINE CORRECTION
# =========================================================
for entry in all_plots_sorted:  # baseline-fitted ΔF/F
    sig = entry["signals"]
    time = sig["time_470"]
    dff  = sig["dF/F"]  # baseline-fitted ΔF/F

    # -----------------------------------------------------
    # Compute sampling rate dynamically
    # -----------------------------------------------------
    dt = np.median(np.diff(time))
    fs = 1.0 / dt

    # -----------------------------------------------------
    # High-pass filter ΔF/F
    # -----------------------------------------------------
    try:
        dff_hp = highpass_filter(dff, HP_CUTOFF_HZ, fs, HP_ORDER)
    except ValueError:
        continue

    plt.figure(figsize=(14, 5))

    # -----------------------------------------------------
    # Align each reward-collect trial
    # -----------------------------------------------------
    for collect_time in entry["collect_times"]:
        if collect_time is None:
            continue

        t_aligned = time - collect_time
        mask = (t_aligned >= -WINDOW_SEC) & (t_aligned <= WINDOW_SEC)
        if mask.sum() < 5:
            continue

        # -------------------------------------------------
        # BASELINE CORRECTION
        # -------------------------------------------------
        baseline_mask = (t_aligned >= BASELINE_START) & (t_aligned <= BASELINE_END)
        if baseline_mask.sum() > 0:
            baseline_mean = np.nanmean(dff_hp[baseline_mask])
        else:
            baseline_mean = 0.0

        dff_corrected = dff_hp[mask] - baseline_mean

        plt.plot(
            t_aligned[mask],
            dff_corrected,
            color="black",
            lw=0.8,
            alpha=0.6
        )

    # Vertical reference line at reward collect
    plt.axvline(0, color="gray", linestyle="--", alpha=0.8)

    plt.title(
        f"Animal {entry['animal_id']} | Genotype {entry['genotype']} | "
        f"Region {entry['region']} | Session {entry['session']} | Exp {entry['experiment']}"
    )
    plt.xlabel("Time relative to reward collect (s)")
    plt.ylabel(f"ΔF/F (high-pass >{HP_CUTOFF_HZ} Hz, baseline-corrected)")
    plt.xlim(-WINDOW_SEC, WINDOW_SEC)
    plt.tight_layout()
    plt.show()

### Reward collect means

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt
from scipy.interpolate import interp1d

# =========================================================
# USER SETTINGS
# =========================================================
HP_CUTOFF_HZ    = 0.03       # Hz, high-pass filter for event-scale drift
HP_ORDER        = 3
WINDOW_SEC      = 10         # seconds before/after reward collect
RESAMPLE_POINTS = 800        # points for uniform alignment
BASELINE_START  = -10        # baseline window start
BASELINE_END    = -5         # baseline window end

# =========================================================
# FILTER FUNCTION
# =========================================================
def highpass_filter(signal, cutoff, fs, order=3):
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    b, a = butter(order, normal_cutoff, btype="high")
    return filtfilt(b, a, signal)

# =========================================================
# PERI-REWARD-COLLECT MEAN + SEM PLOTS WITH BASELINE CORRECTION
# =========================================================
for entry in all_plots_sorted:  # baseline-fitted ΔF/F
    sig = entry["signals"]
    time = sig["time_470"]
    dff  = sig["dF/F"]  # baseline-fitted ΔF/F

    # -----------------------------------------------------
    # Compute sampling rate dynamically
    # -----------------------------------------------------
    dt = np.median(np.diff(time))
    fs = 1.0 / dt

    # -----------------------------------------------------
    # High-pass filter ΔF/F
    # -----------------------------------------------------
    try:
        dff_hp = highpass_filter(dff, HP_CUTOFF_HZ, fs, HP_ORDER)
    except ValueError:
        continue

    plt.figure(figsize=(14, 5))

    aligned_traces = []

    # Common resampled time vector
    t_resampled = np.linspace(-WINDOW_SEC, WINDOW_SEC, RESAMPLE_POINTS)

    # -----------------------------------------------------
    # Align each reward-collect trial
    # -----------------------------------------------------
    for c_time in entry["collect_times"]:
        if c_time is None:
            continue

        t_window = time - c_time
        mask = (t_window >= -WINDOW_SEC) & (t_window <= WINDOW_SEC)
        if mask.sum() < 5:
            continue

        # -------------------------------------------------
        # BASELINE CORRECTION per trial
        # -------------------------------------------------
        baseline_mask = (t_window >= BASELINE_START) & (t_window <= BASELINE_END)
        if baseline_mask.sum() > 0:
            baseline_mean = np.nanmean(dff_hp[baseline_mask])
        else:
            baseline_mean = 0.0

        dff_corrected = dff_hp[mask] - baseline_mean

        # -------------------------------------------------
        # Interpolate corrected trace onto uniform time vector
        # -------------------------------------------------
        interp_func = interp1d(
            t_window[mask],
            dff_corrected,
            kind='linear',
            bounds_error=False,
            fill_value=np.nan
        )
        aligned_traces.append(interp_func(t_resampled))

    # -----------------------------------------------------
    # Compute mean + SEM
    # -----------------------------------------------------
    if aligned_traces:
        aligned_array = np.vstack(aligned_traces)

        mean_trace = np.nanmean(aligned_array, axis=0)
        sem_trace  = np.nanstd(aligned_array, axis=0) / np.sqrt(np.sum(~np.isnan(aligned_array), axis=0))

        # Plot mean + SEM
        plt.plot(t_resampled, mean_trace, color="blue", lw=2, label="Mean ΔF/F")
        plt.fill_between(t_resampled, mean_trace - sem_trace, mean_trace + sem_trace,
                         color="blue", alpha=0.3, label="SEM")

        # Vertical line: reward collect = 0
        plt.axvline(0,  color="gray", linestyle="--", alpha=0.8)

    plt.title(
        f"Animal {entry['animal_id']} | Genotype {entry['genotype']} | "
        f"Region {entry['region']} | Session {entry['session']} | Exp {entry['experiment']}"
    )
    plt.xlabel("Time relative to reward collect (s)")
    plt.ylabel(f"ΔF/F (high-pass >{HP_CUTOFF_HZ} Hz, baseline-corrected)")
    plt.xlim(-WINDOW_SEC, WINDOW_SEC)
    plt.legend()
    plt.tight_layout()
    plt.show()

### Reward collect compiled means

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt
from collections import defaultdict

# =========================================================
# USER SETTINGS
# =========================================================
HP_CUTOFF_HZ    = 0.03     # Hz for event-scale high-pass filter
HP_ORDER        = 3
WINDOW_SEC      = 10       # seconds before/after reward collect
RESAMPLE_POINTS = 800      # points for uniform alignment
BASELINE_START  = -10      # baseline window start
BASELINE_END    = -5       # baseline window end

# =========================================================
# HIGH-PASS FILTER FUNCTION
# =========================================================
def highpass_filter(signal, cutoff, fs, order=3):
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    b, a = butter(order, normal_cutoff, btype="high")
    return filtfilt(b, a, signal)

# =========================================================
# GROUP DATA BY REGION × SESSION × GENOTYPE
# =========================================================
grouped_data = defaultdict(list)

for entry in all_plots_sorted:   # baseline-fitted ΔF/F
    sig = entry["signals"]
    time = sig["time_470"]
    dff  = sig["dF/F"]  # baseline-fitted ΔF/F

    # -----------------------------------------------------
    # Compute sampling rate dynamically
    # -----------------------------------------------------
    dt = np.median(np.diff(time))
    fs = 1.0 / dt

    # -----------------------------------------------------
    # High-pass filter ΔF/F
    # -----------------------------------------------------
    try:
        dff_hp = highpass_filter(dff, HP_CUTOFF_HZ, fs, HP_ORDER)
    except ValueError:
        continue

    t_resampled = np.linspace(-WINDOW_SEC, WINDOW_SEC, RESAMPLE_POINTS)
    aligned_traces = []

    # -----------------------------------------------------
    # Align reward-collect trials with baseline correction
    # -----------------------------------------------------
    for c_time in entry["collect_times"]:
        if c_time is None:
            continue

        t_window = time - c_time
        mask = (t_window >= -WINDOW_SEC) & (t_window <= WINDOW_SEC)
        if not np.any(mask):
            continue

        # Baseline correction per trial
        baseline_mask = (t_window >= BASELINE_START) & (t_window <= BASELINE_END)
        if baseline_mask.sum() > 0:
            baseline_mean = np.nanmean(dff_hp[baseline_mask])
        else:
            baseline_mean = 0.0

        corrected_trace = dff_hp[mask] - baseline_mean

        # Interpolate onto uniform time vector
        interp_func = np.interp(
            t_resampled,
            t_window[mask],
            corrected_trace,
            left=np.nan,
            right=np.nan
        )
        aligned_traces.append(interp_func)

    if aligned_traces:
        aligned_array = np.vstack(aligned_traces)
        mean_trace = np.nanmean(aligned_array, axis=0)
        sem_trace  = np.nanstd(aligned_array, axis=0) / np.sqrt(np.sum(~np.isnan(aligned_array), axis=0))

        key = (entry["region"], entry["session"], entry["genotype"])
        grouped_data[key].append({
            "animal_id": entry["animal_id"],
            "mean_trace": mean_trace,
            "sem_trace": sem_trace
        })

# =========================================================
# PLOT GROUPED MEAN TRACES
# =========================================================
for (region, session, genotype), animal_traces in grouped_data.items():
    plt.figure(figsize=(14, 5))
    t_resampled = np.linspace(-WINDOW_SEC, WINDOW_SEC, RESAMPLE_POINTS)

    # Plot each animal's mean trace
    for animal in animal_traces:
        plt.plot(t_resampled, animal["mean_trace"], lw=2, alpha=0.6, label=f"Animal {animal['animal_id']}")

    # Group mean ± SEM
    all_means = np.array([a["mean_trace"] for a in animal_traces])
    group_mean = np.nanmean(all_means, axis=0)
    group_sem  = np.nanstd(all_means, axis=0) / np.sqrt(len(animal_traces))

    plt.plot(t_resampled, group_mean, color="black", lw=3, label="Group mean")
    plt.fill_between(t_resampled, group_mean - group_sem, group_mean + group_sem,
                     color="gray", alpha=0.3, label="Group SEM")

    # Vertical reference line at reward collect = 0
    plt.axvline(0, color="gray", linestyle="--", alpha=0.8)

    plt.title(f"Region {region} | Session {session} | Genotype {genotype}")
    plt.xlabel("Time relative to reward collect (s)")
    plt.ylabel(f"ΔF/F (high-pass >{HP_CUTOFF_HZ} Hz, baseline-corrected)")
    plt.xlim(-WINDOW_SEC, WINDOW_SEC)
    plt.legend()
    plt.tight_layout()
    plt.show()

### Reward_summary

In [ ]:
import numpy as np
from scipy.signal import butter, filtfilt
from scipy.interpolate import interp1d
import pandas as pd
import os

# =========================================================
# USER SETTINGS
# =========================================================
HP_CUTOFF_HZ    = 0.03       # Hz, high-pass filter for event-scale drift
HP_ORDER        = 3
WINDOW_SEC      = 10         # seconds before/after event
RESAMPLE_POINTS = 800        # points for uniform alignment
BASELINE_START  = -10        # baseline window start
BASELINE_END    = -5         # baseline window end

OUTPUT_FOLDER = "/Users/gsw512/Documents/E1.2_MGK/females"
OUTPUT_FILE_REWARD = "Rewards_summary.csv"
OUTPUT_FILE_COLLECT = "RewardCollect_summary.csv"

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# =========================================================
# HIGH-PASS FILTER FUNCTION
# =========================================================
def highpass_filter(signal, cutoff, fs, order=3):
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    b, a = butter(order, normal_cutoff, btype="high")
    return filtfilt(b, a, signal)

# =========================================================
# GENOTYPE ORDER MAP
# =========================================================
genotype_order = {"WT": 0, "KI": 1}

# =========================================================
# FUNCTION TO COMPUTE MEAN TRACE WITH BASELINE CORRECTION
# =========================================================
def compute_mean_trace(time, signal, event_times):
    dt = np.median(np.diff(time))
    fs = 1.0 / dt
    try:
        signal_hp = highpass_filter(signal, HP_CUTOFF_HZ, fs, HP_ORDER)
    except ValueError:
        return None

    t_resampled = np.linspace(-WINDOW_SEC, WINDOW_SEC, RESAMPLE_POINTS)
    aligned_traces = []

    for e_time in event_times:
        if e_time is None:
            continue

        t_window = time - e_time
        mask = (t_window >= -WINDOW_SEC) & (t_window <= WINDOW_SEC)
        if not np.any(mask):
            continue

        # Baseline correction
        baseline_mask = (t_window >= BASELINE_START) & (t_window <= BASELINE_END)
        baseline_mean = np.nanmean(signal_hp[baseline_mask]) if baseline_mask.sum() > 0 else 0.0
        corrected_trace = signal_hp[mask] - baseline_mean

        # Interpolate onto uniform time vector
        interp_trace = np.interp(
            t_resampled,
            t_window[mask],
            corrected_trace,
            left=np.nan,
            right=np.nan
        )
        aligned_traces.append(interp_trace)

    if aligned_traces:
        return np.nanmean(np.vstack(aligned_traces), axis=0)
    else:
        return None

# =========================================================
# FUNCTION TO EXPORT CSV
# =========================================================
def export_csv(all_plots, event_key, output_file):
    all_data_rows = []
    # First row: time vector
    t_resampled = np.linspace(-WINDOW_SEC, WINDOW_SEC, RESAMPLE_POINTS)
    all_data_rows.append(["time"] + t_resampled.tolist())

    data_rows_for_sort = []

    for entry in all_plots:
        sig = entry["signals"]
        time = sig["time_470"]
        dff  = sig["dF/F"]

        mean_trace = compute_mean_trace(time, dff, entry[event_key])
        if mean_trace is not None:
            row = [entry["region"], entry["session"], entry["genotype"], entry["animal_id"]] + mean_trace.tolist()
            data_rows_for_sort.append(row)

    # Sort by region → session → genotype (WT first)
    data_rows_sorted = sorted(
        data_rows_for_sort,
        key=lambda x: (x[0], x[1], genotype_order.get(x[2], 99))
    )

    all_data_rows += data_rows_sorted
    csv_path = os.path.join(OUTPUT_FOLDER, output_file)
    pd.DataFrame(all_data_rows).to_csv(csv_path, index=False, header=False)
    print(f"✅ Saved {event_key} mean traces to {csv_path}")

# =========================================================
# EXPORT BOTH CSVs
# =========================================================
export_csv(all_plots_sorted, "reward_times", OUTPUT_FILE_REWARD)
export_csv(all_plots_sorted, "collect_times", OUTPUT_FILE_COLLECT)

In [ ]:
# =========================================================
# AUC
# =========================================================

import numpy as np
from scipy.signal import butter, filtfilt
from scipy.interpolate import interp1d
import pandas as pd
import os

# =========================================================
# USER SETTINGS
# =========================================================
HP_CUTOFF_HZ    = 0.03       # Hz, high-pass filter for event-scale drift
HP_ORDER        = 3
WINDOW_SEC      = 10         # seconds before/after reward
RESAMPLE_POINTS = 800        # points for uniform alignment
BASELINE_START  = -10        # baseline window start
BASELINE_END    = -5         # baseline window end
CS_START        = -5         # CS interval start
CS_END          = 0          # CS interval end
US_START        = 0          # US interval start
US_END          = 5          # US interval end

OUTPUT_FOLDER   = "/Users/gsw512/Documents/E1.2_MGK/females"
OUTPUT_FILE     = "Reward_AUC_summary.csv"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# =========================================================
# HIGH-PASS FILTER FUNCTION
# =========================================================
def highpass_filter(signal, cutoff, fs, order=3):
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    b, a = butter(order, normal_cutoff, btype="high")
    return filtfilt(b, a, signal)

# =========================================================
# GENOTYPE ORDER
# =========================================================
genotype_order = {"WT": 0, "KI": 1}

# =========================================================
# STORE AUC PER TRIAL
# =========================================================
auc_results = []

# Uniform time vector for interpolation
t_resampled = np.linspace(-WINDOW_SEC, WINDOW_SEC, RESAMPLE_POINTS)

for entry in all_plots_sorted:  # baseline-fitted ΔF/F
    sig = entry["signals"]
    time = sig["time_470"]
    dff  = sig["dF/F"]

    # Compute sampling rate
    dt = np.median(np.diff(time))
    fs = 1.0 / dt

    # High-pass filter
    try:
        dff_hp = highpass_filter(dff, HP_CUTOFF_HZ, fs, HP_ORDER)
    except ValueError:
        continue

    for r_time in entry["reward_times"]:
        if r_time is None:
            continue

        t_aligned = time - r_time
        mask = (t_aligned >= -WINDOW_SEC) & (t_aligned <= WINDOW_SEC)
        if mask.sum() < 5:
            continue

        # Baseline correction
        baseline_mask = (t_aligned >= BASELINE_START) & (t_aligned <= BASELINE_END)
        baseline_mean = np.nanmean(dff_hp[baseline_mask]) if baseline_mask.sum() > 0 else 0.0
        dff_corrected = dff_hp[mask] - baseline_mean

        # Interpolate onto uniform vector
        interp_func = interp1d(
            t_aligned[mask],
            dff_corrected,
            kind='linear',
            bounds_error=False,
            fill_value=np.nan
        )
        aligned_trace = interp_func(t_resampled)

        # Compute AUCs
        cs_mask = (t_resampled >= CS_START) & (t_resampled <= CS_END)
        us_mask = (t_resampled >= US_START) & (t_resampled <= US_END)

        cs_auc = np.trapz(aligned_trace[cs_mask], t_resampled[cs_mask])
        us_auc = np.trapz(aligned_trace[us_mask], t_resampled[us_mask])

        # Compute CS/US AUC ratio (protect against divide-by-zero)
        auc_log_ratio = np.log(
            (np.abs(cs_auc) + 1e-6) /
            (np.abs(us_auc) + 1e-6)
        )

        auc_results.append({
            "animal_id": entry["animal_id"],
            "region": entry["region"],
            "session": entry["session"],
            "genotype": entry["genotype"],
            "CS_AUC": cs_auc,
            "US_AUC": us_auc,
            "CS_US_AUC_ratio": auc_log_ratio
        })

# =========================================================
# COMPUTE MEAN AUC PER ANIMAL
# =========================================================
df_auc = pd.DataFrame(auc_results)
df_mean = df_auc.groupby(
    ["region", "session", "genotype", "animal_id"],
    as_index=False
)[["CS_AUC", "US_AUC", "CS_US_AUC_ratio"]].mean()

# =========================================================
# SORT BY REGION → SESSION → GENOTYPE (WT first)
# =========================================================
df_mean["genotype_order"] = df_mean["genotype"].map(genotype_order)
df_mean_sorted = df_mean.sort_values(["region", "session", "genotype_order"]).drop(columns="genotype_order")

# =========================================================
# SAVE CSV
# =========================================================
csv_path = os.path.join(OUTPUT_FOLDER, OUTPUT_FILE)
df_mean_sorted.to_csv(csv_path, index=False)

print(f"✅ Saved Reward AUC summary (CS & US intervals) to {csv_path}")

In [ ]:
# =========================================================
# PEAK AMPLITUDE (REWARD)
# =========================================================

import numpy as np
from scipy.signal import butter, filtfilt
from scipy.interpolate import interp1d
import pandas as pd
import os

# =========================================================
# USER SETTINGS
# =========================================================
HP_CUTOFF_HZ     = 0.03       # Hz, high-pass filter for event-scale drift
HP_ORDER         = 3
WINDOW_SEC       = 10         # seconds before/after reward
RESAMPLE_POINTS  = 800        # points for uniform alignment

BASELINE_START   = -10        # baseline window start
BASELINE_END     = -5         # baseline window end

CS_START         = -5         # CS peak window start
CS_END           = 0         # CS peak window end
US_START         = 0         # US peak window start
US_END           = 5          # US peak window end

OUTPUT_FOLDER    = "/Users/gsw512/Documents/E1.2_MGK/females"
OUTPUT_FILE      = "Reward_Peak_summary.csv"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# =========================================================
# HIGH-PASS FILTER FUNCTION
# =========================================================
def highpass_filter(signal, cutoff, fs, order=3):
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    b, a = butter(order, normal_cutoff, btype="high")
    return filtfilt(b, a, signal)

# =========================================================
# GENOTYPE ORDER
# =========================================================
genotype_order = {"WT": 0, "KI": 1}

# =========================================================
# STORE PEAKS PER TRIAL
# =========================================================
peak_results = []

# Uniform time vector
t_resampled = np.linspace(-WINDOW_SEC, WINDOW_SEC, RESAMPLE_POINTS)

for entry in all_plots_sorted:
    sig = entry["signals"]
    time = sig["time_470"]
    dff  = sig["dF/F"]

    # Sampling rate
    dt = np.median(np.diff(time))
    fs = 1.0 / dt

    # High-pass filter
    try:
        dff_hp = highpass_filter(dff, HP_CUTOFF_HZ, fs, HP_ORDER)
    except ValueError:
        continue

    for r_time in entry["reward_times"]:
        if r_time is None:
            continue

        t_aligned = time - r_time
        mask = (t_aligned >= -WINDOW_SEC) & (t_aligned <= WINDOW_SEC)
        if mask.sum() < 5:
            continue

        # -------------------------------------------------
        # BASELINE CORRECTION
        # -------------------------------------------------
        baseline_mask = (t_aligned >= BASELINE_START) & (t_aligned <= BASELINE_END)
        baseline_mean = np.nanmean(dff_hp[baseline_mask]) if baseline_mask.sum() > 0 else 0.0
        dff_corrected = dff_hp[mask] - baseline_mean

        # -------------------------------------------------
        # INTERPOLATE
        # -------------------------------------------------
        interp_func = interp1d(
            t_aligned[mask],
            dff_corrected,
            kind="linear",
            bounds_error=False,
            fill_value=np.nan
        )
        aligned_trace = interp_func(t_resampled)

        # -------------------------------------------------
        # PEAK AMPLITUDES
        # -------------------------------------------------
        cs_mask = (t_resampled >= CS_START) & (t_resampled <= CS_END)
        us_mask = (t_resampled >= US_START) & (t_resampled <= US_END)

        cs_peak = np.nanmax(aligned_trace[cs_mask])
        us_peak = np.nanmax(aligned_trace[us_mask])

        peak_results.append({
            "animal_id": entry["animal_id"],
            "region": entry["region"],
            "session": entry["session"],
            "genotype": entry["genotype"],
            "CS_peak": cs_peak,
            "US_peak": us_peak
        })

# =========================================================
# MEAN PEAK PER ANIMAL
# =========================================================
df_peaks = pd.DataFrame(peak_results)

df_mean = df_peaks.groupby(
    ["region", "session", "genotype", "animal_id"],
    as_index=False
)[["CS_peak", "US_peak"]].mean()

# =========================================================
# SORT BY REGION → SESSION → GENOTYPE (WT first)
# =========================================================
df_mean["genotype_order"] = df_mean["genotype"].map(genotype_order)
df_mean_sorted = (
    df_mean
    .sort_values(["region", "session", "genotype_order"])
    .drop(columns="genotype_order")
)

# =========================================================
# SAVE CSV
# =========================================================
csv_path = os.path.join(OUTPUT_FOLDER, OUTPUT_FILE)
df_mean_sorted.to_csv(csv_path, index=False)

print(f"✅ Saved Reward peak amplitudes (CS & US) to {csv_path}")

### Reward collect_summary

In [ ]:
import numpy as np
from scipy.signal import butter, filtfilt
from scipy.interpolate import interp1d
import pandas as pd
import os

# =========================================================
# USER SETTINGS
# =========================================================
HP_CUTOFF_HZ    = 0.03       # Hz, high-pass filter for event-scale drift
HP_ORDER        = 3
WINDOW_SEC      = 10         # seconds before/after reward collect
RESAMPLE_POINTS = 800        # points for uniform alignment

OUTPUT_FOLDER = "/Users/gsw512/Documents/E1.2_MGK/females"
OUTPUT_FILE   = "Collects_summary.csv"

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# =========================================================
# HIGH-PASS FILTER FUNCTION
# =========================================================
def highpass_filter(signal, cutoff, fs, order=3):
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    b, a = butter(order, normal_cutoff, btype="high")
    return filtfilt(b, a, signal)

# =========================================================
# GENOTYPE ORDER MAP
# =========================================================
genotype_order = {"WT": 0, "KI": 1}

# =========================================================
# STORE DATA
# =========================================================
all_data_rows = []

# Resampled time vector as first row
t_resampled = np.linspace(-WINDOW_SEC, WINDOW_SEC, RESAMPLE_POINTS)
all_data_rows.append(["time"] + t_resampled.tolist())

# =========================================================
# COMPUTE PERI-REWARD-COLLECT MEAN TRACES
# =========================================================
data_rows_for_sort = []

for entry in all_plots_sorted:  # baseline-fitted ΔF/F
    sig = entry["signals"]
    time = sig["time_470"]
    dff  = sig["dF/F"]  # baseline-fitted ΔF/F

    # Compute sampling rate dynamically
    dt = np.median(np.diff(time))
    fs = 1.0 / dt

    # High-pass filter ΔF/F
    try:
        dff_hp = highpass_filter(dff, HP_CUTOFF_HZ, fs, HP_ORDER)
    except ValueError:
        continue

    aligned_traces = []

    # Align each reward-collect trial
    for c_time in entry["collect_times"]:
        if c_time is None:
            continue

        t_window = time - c_time
        mask = (t_window >= -WINDOW_SEC) & (t_window <= WINDOW_SEC)
        if not np.any(mask):
            continue

        # Linear interpolation, no extrapolation
        interp_func = interp1d(
            t_window[mask],
            dff_hp[mask],
            kind='linear',
            bounds_error=False,
            fill_value=np.nan
        )
        aligned_traces.append(interp_func(t_resampled))

    # Compute mean trace
    if aligned_traces:
        aligned_array = np.vstack(aligned_traces)
        mean_trace = np.nanmean(aligned_array, axis=0)

        # Prepare row: [region, session, genotype, animal_id, mean values...]
        row = [entry["region"], entry["session"], entry["genotype"], entry["animal_id"]] + mean_trace.tolist()
        data_rows_for_sort.append(row)

# =========================================================
# SORT ROWS BY REGION → SESSION → GENOTYPE (WT first)
# =========================================================
data_rows_sorted = sorted(
    data_rows_for_sort,
    key=lambda x: (x[0], x[1], genotype_order.get(x[2], 99))  # unknown genotypes go last
)

# =========================================================
# COMBINE WITH TIME ROW AND SAVE CSV
# =========================================================
all_data_rows += data_rows_sorted
csv_path = os.path.join(OUTPUT_FOLDER, OUTPUT_FILE)
df = pd.DataFrame(all_data_rows)
df.to_csv(csv_path, index=False, header=False)

print(f"✅ Saved mean reward-collect traces (sorted by region → session → genotype WT→KI) to {csv_path}")

In [ ]:
# =========================================================
# AUC
# =========================================================

import numpy as np
from scipy.signal import butter, filtfilt
from scipy.interpolate import interp1d
import pandas as pd
import os

# =========================================================
# USER SETTINGS
# =========================================================
HP_CUTOFF_HZ   = 0.03       # Hz for event-scale high-pass
HP_ORDER       = 3
WINDOW_SEC     = 10         # seconds before/after reward collect
RESAMPLE_POINTS = 800       # points for uniform alignment
BASELINE_START = -10        # baseline window start
BASELINE_END   = -5         # baseline window end
AUC_START      = -1         # start of integration window
AUC_END        = 5          # end of integration window

OUTPUT_FOLDER  = "/Users/gsw512/Documents/E1.2_MGK/females"
OUTPUT_FILE    = "RewardCollect_AUC_summary.csv"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# =========================================================
# HIGH-PASS FILTER FUNCTION
# =========================================================
def highpass_filter(signal, cutoff, fs, order=3):
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    b, a = butter(order, normal_cutoff, btype="high")
    return filtfilt(b, a, signal)

# =========================================================
# GENOTYPE ORDER
# =========================================================
genotype_order = {"WT": 0, "KI": 1}

# =========================================================
# STORE AUC PER TRIAL
# =========================================================
auc_results = []

t_resampled = np.linspace(-WINDOW_SEC, WINDOW_SEC, RESAMPLE_POINTS)

for entry in all_plots_sorted:  # baseline-fitted ΔF/F
    sig = entry["signals"]
    time = sig["time_470"]
    dff  = sig["dF/F"]

    # Compute sampling rate
    dt = np.median(np.diff(time))
    fs = 1.0 / dt

    # High-pass filter
    try:
        dff_hp = highpass_filter(dff, HP_CUTOFF_HZ, fs, HP_ORDER)
    except ValueError:
        continue

    for collect_time in entry["collect_times"]:
        if collect_time is None:
            continue

        t_aligned = time - collect_time
        mask = (t_aligned >= -WINDOW_SEC) & (t_aligned <= WINDOW_SEC)
        if mask.sum() < 5:
            continue

        # Baseline correction
        baseline_mask = (t_aligned >= BASELINE_START) & (t_aligned <= BASELINE_END)
        baseline_mean = np.nanmean(dff_hp[baseline_mask]) if baseline_mask.sum() > 0 else 0.0
        dff_corrected = dff_hp[mask] - baseline_mean

        # Interpolate onto uniform vector
        interp_func = interp1d(
            t_aligned[mask],
            dff_corrected,
            kind='linear',
            bounds_error=False,
            fill_value=np.nan
        )
        aligned_trace = interp_func(t_resampled)

        # AUC mask
        auc_mask = (t_resampled >= AUC_START) & (t_resampled <= AUC_END)
        auc_value = np.trapz(aligned_trace[auc_mask], t_resampled[auc_mask])

        auc_results.append({
            "animal_id": entry["animal_id"],
            "region": entry["region"],
            "session": entry["session"],
            "genotype": entry["genotype"],
            "AUC": auc_value
        })

# =========================================================
# COMPUTE MEAN AUC PER ANIMAL
# =========================================================
df_auc = pd.DataFrame(auc_results)
df_mean = df_auc.groupby(
    ["region", "session", "genotype", "animal_id"],
    as_index=False
)["AUC"].mean()

# =========================================================
# SORT BY REGION → SESSION → GENOTYPE (WT first)
# =========================================================
df_mean["genotype_order"] = df_mean["genotype"].map(genotype_order)
df_mean_sorted = df_mean.sort_values(["region", "session", "genotype_order"]).drop(columns="genotype_order")

# =========================================================
# SAVE CSV
# =========================================================
csv_path = os.path.join(OUTPUT_FOLDER, OUTPUT_FILE)
df_mean_sorted.to_csv(csv_path, index=False)

print(f"✅ Saved Reward Collect AUC summary to {csv_path}")

In [ ]:
# =========================================================
# PEAK AMPLITUDE (Reward Collect)
# =========================================================

import numpy as np
from scipy.signal import butter, filtfilt
from scipy.interpolate import interp1d
import pandas as pd
import os

# =========================================================
# USER SETTINGS
# =========================================================
HP_CUTOFF_HZ    = 0.03
HP_ORDER        = 3
WINDOW_SEC      = 10
RESAMPLE_POINTS = 800
BASELINE_START  = -10
BASELINE_END    = -5

PEAK_START      = -1     # same interval as AUC
PEAK_END        = 5

OUTPUT_FOLDER   = "/Users/gsw512/Documents/E1.2_MGK/females"
OUTPUT_FILE     = "RewardCollect_Peak_summary.csv"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# =========================================================
# HIGH-PASS FILTER FUNCTION
# =========================================================
def highpass_filter(signal, cutoff, fs, order=3):
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    b, a = butter(order, normal_cutoff, btype="high")
    return filtfilt(b, a, signal)

# =========================================================
# GENOTYPE ORDER
# =========================================================
genotype_order = {"WT": 0, "KI": 1}

# =========================================================
# STORE PEAK PER TRIAL
# =========================================================
peak_results = []

t_resampled = np.linspace(-WINDOW_SEC, WINDOW_SEC, RESAMPLE_POINTS)

for entry in all_plots_sorted:
    sig = entry["signals"]
    time = sig["time_470"]
    dff  = sig["dF/F"]

    # Sampling rate
    dt = np.median(np.diff(time))
    fs = 1.0 / dt

    # High-pass filter
    try:
        dff_hp = highpass_filter(dff, HP_CUTOFF_HZ, fs, HP_ORDER)
    except ValueError:
        continue

    for collect_time in entry["collect_times"]:
        if collect_time is None:
            continue

        t_aligned = time - collect_time
        mask = (t_aligned >= -WINDOW_SEC) & (t_aligned <= WINDOW_SEC)
        if mask.sum() < 5:
            continue

        # -------------------------------------------------
        # BASELINE CORRECTION
        # -------------------------------------------------
        baseline_mask = (t_aligned >= BASELINE_START) & (t_aligned <= BASELINE_END)
        baseline_mean = np.nanmean(dff_hp[baseline_mask]) if baseline_mask.sum() > 0 else 0.0
        dff_corrected = dff_hp[mask] - baseline_mean

        # -------------------------------------------------
        # INTERPOLATE
        # -------------------------------------------------
        interp_func = interp1d(
            t_aligned[mask],
            dff_corrected,
            kind="linear",
            bounds_error=False,
            fill_value=np.nan
        )
        aligned_trace = interp_func(t_resampled)

        # -------------------------------------------------
        # PEAK AMPLITUDE
        # -------------------------------------------------
        peak_mask = (t_resampled >= PEAK_START) & (t_resampled <= PEAK_END)
        peak_value = np.nanmax(aligned_trace[peak_mask])

        peak_results.append({
            "animal_id": entry["animal_id"],
            "region": entry["region"],
            "session": entry["session"],
            "genotype": entry["genotype"],
            "PeakAmplitude": peak_value
        })

# =========================================================
# MEAN PEAK PER ANIMAL
# =========================================================
df_peak = pd.DataFrame(peak_results)

df_mean = df_peak.groupby(
    ["region", "session", "genotype", "animal_id"],
    as_index=False
)["PeakAmplitude"].mean()

# =========================================================
# SORT (Region → Session → Genotype)
# =========================================================
df_mean["genotype_order"] = df_mean["genotype"].map(genotype_order)
df_mean_sorted = (
    df_mean
    .sort_values(["region", "session", "genotype_order"])
    .drop(columns="genotype_order")
)

# =========================================================
# SAVE CSV
# =========================================================
csv_path = os.path.join(OUTPUT_FOLDER, OUTPUT_FILE)
df_mean_sorted.to_csv(csv_path, index=False)

print(f"✅ Saved Reward Collect PEAK summary to {csv_path}")

## (470-415)/mean(470baseline - 415baseline)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ----------- COLLECT BASELINE-NORMALIZED ΔF/F TRACES -----------

all_plots_baseline = []

for exp_folder in input_path.iterdir():
    if not exp_folder.is_dir():
        continue

    metadata = parse_exp_folder(exp_folder.name)

    for sub_name in subfolders:
        if "habituation" in sub_name.lower():
            continue

        sub_path = exp_folder / sub_name
        if not sub_path.exists():
            continue

        session_files = collect_session_files(sub_path)

        for beh_file in session_files["behavior"]:
            try:
                animal_id = get_animal_id_from_behavior_csv(beh_file)
            except Exception as e:
                print(f"❌ {beh_file.name}: {e}")
                continue

            photometry_file = (
                session_files["photometry"][0]
                if session_files["photometry"]
                else None
            )

            fibers = fiber_map.get(animal_id, {})
            genotype = genotype_map.get(animal_id, "Unknown")

            if photometry_file is None or not fibers:
                continue

            ttl_file = get_ttl_for_animal(
                session_files["ttl"],
                animal_id,
                metadata["animal_A"],
                metadata["animal_B"]
            )

            session_start = get_session_start([ttl_file]) if ttl_file else 0.0

            signals = extract_photometry_signals(
                photometry_file,
                fibers,
                session_start
            )

            events = extract_behavior_events(
                beh_file,
                session_start
            )

            if events is None or events.empty:
                continue

            # Filter rewards
            reward_events = events[events["Item_Name"] == "reward"]
            if reward_events.empty:
                continue

            max_reward_time = reward_events["Evnt_Time"].max()

            for region, sig in signals.items():
                # ---- Chop signals to [-900, last reward] ----
                for led in ["470", "415"]:
                    tkey = f"time_{led}"
                    mask = (sig[tkey] >= -900) & (sig[tkey] <= max_reward_time)
                    sig[tkey] = sig[tkey][mask]
                    sig[led] = sig[led][mask]

                # ---- Compute baseline ΔF/F ----
                baseline_mask = sig["time_470"] < 0  # baseline is all time < 0
                F470_baseline = np.mean(sig["470"][baseline_mask])
                F415_baseline = np.mean(sig["415"][baseline_mask])

                # Avoid division by zero
                sig["dF/F_baseline"] = np.where(
                    (F470_baseline - F415_baseline) != 0,
                    (sig["470"] - sig["415"]) / (F470_baseline - F415_baseline),
                    np.nan
                )

                # ---- Extract reward-collect times ----
                collect_times = []
                for r_time in reward_events["Evnt_Time"].values:
                    post_reward = events[
                        (events["Evnt_Time"] > r_time) &
                        (events["Item_Name"] == "ITI_head_entry")
                    ]
                    if not post_reward.empty:
                        collect_times.append(post_reward.iloc[0]["Evnt_Time"])
                    else:
                        collect_times.append(None)

                # ---- Store plot entry ----
                all_plots_baseline.append({
                    "animal_id"     : animal_id,
                    "genotype"      : genotype,
                    "experiment"    : metadata["experiment"],
                    "session"       : sub_name,
                    "region"        : region,
                    "signals"       : sig,
                    "reward_times"  : reward_events["Evnt_Time"].values,
                    "collect_times" : collect_times
                })

In [ ]:
# ----------- GLOBAL SORT (ACROSS ALL SESSIONS) -----------

def animal_sort_key(entry):
    aid = entry["animal_id"]
    return int(aid) if str(aid).isdigit() else str(aid)

all_plots_baseline_sorted = sorted(all_plots_baseline, key=animal_sort_key)

In [ ]:
# ----------- PLOT BASELINE-NORMALIZED ΔF/F (SORTED) -----------

for entry in all_plots_baseline_sorted:
    sig = entry["signals"]

    plt.figure(figsize=(24, 5))

    # ΔF/F baseline-normalized
    plt.plot(
        sig["time_470"],
        sig["dF/F_baseline"],
        label="ΔF/F (baseline)",
        color="black",
        lw=0.8
    )

    # Reward delivery (red)
    for i, t in enumerate(entry["reward_times"]):
        plt.axvline(
            t,
            color="red",
            linestyle="--",
            alpha=0.6,
            label="Reward" if i == 0 else None
        )

    # Reward collect (purple)
    for i, t in enumerate(entry["collect_times"]):
        if t is None:
            continue
        plt.axvline(
            t,
            color="purple",
            linestyle=":",
            alpha=0.8,
            label="Reward collect" if i == 0 else None
        )

    plt.title(
        f"Animal {entry['animal_id']} | "
        f"Genotype {entry['genotype']} | "
        f"Region {entry['region']} | "
        f"Session {entry['session']} | "
        f"Exp {entry['experiment']}"
    )

    plt.xlabel("Time (s, chopped)")
    plt.ylabel("ΔF/F (baseline)")
    plt.legend()
    plt.tight_layout()
    plt.show()

## ( (470-415) - mean(470-415)baseline ) / mean(470-415)baseline

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ----------- COLLECT BASELINE-NORMALIZED ΔF/F TRACES (SAFE FORMULA) -----------

all_plots_baseline = []

for exp_folder in input_path.iterdir():
    if not exp_folder.is_dir():
        continue

    metadata = parse_exp_folder(exp_folder.name)

    for sub_name in subfolders:
        if "habituation" in sub_name.lower():
            continue

        sub_path = exp_folder / sub_name
        if not sub_path.exists():
            continue

        session_files = collect_session_files(sub_path)

        for beh_file in session_files["behavior"]:
            try:
                animal_id = get_animal_id_from_behavior_csv(beh_file)
            except Exception as e:
                print(f"❌ {beh_file.name}: {e}")
                continue

            photometry_file = (
                session_files["photometry"][0] if session_files["photometry"] else None
            )

            fibers = fiber_map.get(animal_id, {})
            genotype = genotype_map.get(animal_id, "Unknown")

            if photometry_file is None or not fibers:
                continue

            ttl_file = get_ttl_for_animal(
                session_files["ttl"],
                animal_id,
                metadata["animal_A"],
                metadata["animal_B"]
            )

            session_start = get_session_start([ttl_file]) if ttl_file else 0.0

            signals = extract_photometry_signals(
                photometry_file,
                fibers,
                session_start
            )

            events = extract_behavior_events(
                beh_file,
                session_start
            )

            if events is None or events.empty:
                continue

            # Filter rewards
            reward_events = events[events["Item_Name"] == "reward"]
            if reward_events.empty:
                continue

            max_reward_time = reward_events["Evnt_Time"].max()

            for region, sig in signals.items():
                # ---- Chop signals to [-900, last reward] ----
                for led in ["470", "415"]:
                    tkey = f"time_{led}"
                    mask = (sig[tkey] >= -900) & (sig[tkey] <= max_reward_time)
                    sig[tkey] = sig[tkey][mask]
                    sig[led] = sig[led][mask]

                # ---- Compute baseline ΔF/F safely ----
                baseline_mask = sig["time_470"] < 0  # baseline is all time < 0
                F_diff = sig["470"] - sig["415"]
                F0 = F_diff[baseline_mask].mean()  # baseline of subtracted signal

                # Avoid division by zero
                epsilon = 1e-6
                sig["dF/F_baseline"] = (F_diff - F0) / max(F0, epsilon)

                # ---- Extract reward-collect times ----
                collect_times = []
                for r_time in reward_events["Evnt_Time"].values:
                    post_reward = events[
                        (events["Evnt_Time"] > r_time) &
                        (events["Item_Name"] == "ITI_head_entry")
                    ]
                    if not post_reward.empty:
                        collect_times.append(post_reward.iloc[0]["Evnt_Time"])
                    else:
                        collect_times.append(None)

                # ---- Store plot entry ----
                all_plots_baseline.append({
                    "animal_id"     : animal_id,
                    "genotype"      : genotype,
                    "experiment"    : metadata["experiment"],
                    "session"       : sub_name,
                    "region"        : region,
                    "signals"       : sig,
                    "reward_times"  : reward_events["Evnt_Time"].values,
                    "collect_times" : collect_times
                })

In [ ]:
# ----------- GLOBAL SORT (ACROSS ALL SESSIONS) -----------

def animal_sort_key(entry):
    aid = entry["animal_id"]
    return int(aid) if str(aid).isdigit() else str(aid)

all_plots_baseline_sorted = sorted(all_plots_baseline, key=animal_sort_key)

In [ ]:
# ----------- PLOT BASELINE-NORMALIZED ΔF/F (SORTED) -----------

for entry in all_plots_baseline_sorted:
    sig = entry["signals"]

    plt.figure(figsize=(24, 5))

    # ΔF/F baseline-normalized
    plt.plot(
        sig["time_470"],
        sig["dF/F_baseline"],
        label="ΔF/F (baseline)",
        color="black",
        lw=0.8
    )

    # Reward delivery (red)
    for i, t in enumerate(entry["reward_times"]):
        plt.axvline(
            t,
            color="red",
            linestyle="--",
            alpha=0.6,
            label="Reward" if i == 0 else None
        )

    # Reward collect (purple)
    for i, t in enumerate(entry["collect_times"]):
        if t is None:
            continue
        plt.axvline(
            t,
            color="purple",
            linestyle=":",
            alpha=0.8,
            label="Reward collect" if i == 0 else None
        )

    plt.title(
        f"Animal {entry['animal_id']} | "
        f"Genotype {entry['genotype']} | "
        f"Region {entry['region']} | "
        f"Session {entry['session']} | "
        f"Exp {entry['experiment']}"
    )

    plt.xlabel("Time (s, chopped)")
    plt.ylabel("ΔF/F (baseline)")
    plt.legend()
    plt.tight_layout()
    plt.show()

## Baseline normalisation + high pass filter

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.signal import butter, filtfilt

# ----------- USER SETTINGS -----------
# High-pass filter cutoff frequency (Hz)
highpass_cutoff = 0.03  # e.g., remove <0.01 Hz drifts
sampling_rate = 400      # Hz, adjust to your actual sampling rate

# ----------- FILTER FUNCTION -----------
def highpass_filter(signal, cutoff, fs, order=3):
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    b, a = butter(order, normal_cutoff, btype='high', analog=False)
    return filtfilt(b, a, signal)

# ----------- APPLY HIGH-PASS FILTER TO ALL BASELINE ΔF/F TRACES -----------

all_plots_filtered = []

for entry in all_plots_baseline_sorted:  # reuse the already baseline-normalized traces
    sig = entry["signals"].copy()  # avoid modifying original

    # Apply high-pass filter
    sig["dF/F_filtered"] = highpass_filter(
        sig["dF/F_baseline"], 
        cutoff=highpass_cutoff, 
        fs=sampling_rate
    )

    # Store filtered signal
    all_plots_filtered.append({
        "animal_id"     : entry["animal_id"],
        "genotype"      : entry["genotype"],
        "experiment"    : entry["experiment"],
        "session"       : entry["session"],
        "region"        : entry["region"],
        "signals"       : sig,
        "reward_times"  : entry["reward_times"],
        "collect_times" : entry["collect_times"]
    })

# ----------- PLOT HIGH-PASS FILTERED ΔF/F (SORTED) -----------

for entry in all_plots_filtered:
    sig = entry["signals"]

    plt.figure(figsize=(24, 5))

    # High-pass filtered ΔF/F
    plt.plot(
        sig["time_470"],
        sig["dF/F_filtered"],
        label=f"ΔF/F baseline (high-pass {highpass_cutoff} Hz)",
        color="black",
        lw=0.8
    )

    # Reward delivery (red)
    for i, t in enumerate(entry["reward_times"]):
        plt.axvline(
            t,
            color="red",
            linestyle="--",
            alpha=0.6,
            label="Reward" if i == 0 else None
        )

    # Reward collect (purple)
    for i, t in enumerate(entry["collect_times"]):
        if t is None:
            continue
        plt.axvline(
            t,
            color="purple",
            linestyle=":",
            alpha=0.8,
            label="Reward collect" if i == 0 else None
        )

    plt.title(
        f"Animal {entry['animal_id']} | "
        f"Genotype {entry['genotype']} | "
        f"Region {entry['region']} | "
        f"Session {entry['session']} | "
        f"Exp {entry['experiment']}"
    )

    plt.xlim(-900)

    plt.xlabel("Time (s, chopped)")
    plt.ylabel(f"ΔF/F (high-pass >{highpass_cutoff} Hz)")
    plt.legend()
    plt.tight_layout()
    plt.show()

### Reward

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.signal import butter, filtfilt

# ----------- USER SETTINGS -----------
highpass_cutoff = 0.03  # Hz
sampling_rate = 400     # Hz
window_sec = 10         # seconds before and after reward

# ----------- FILTER FUNCTION -----------
def highpass_filter(signal, cutoff, fs, order=3):
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    b, a = butter(order, normal_cutoff, btype='high', analog=False)
    return filtfilt(b, a, signal)

# ----------- PERI-REWARD PLOTS (0 = reward, no collect) -----------
for entry in all_plots_baseline_sorted:
    sig = entry["signals"].copy()
    
    # Apply high-pass filter
    sig["dF/F_filtered"] = highpass_filter(sig["dF/F_baseline"], highpass_cutoff, sampling_rate)
    
    plt.figure(figsize=(14, 5))
    
    for r_time in entry["reward_times"]:
        # Align time so reward = 0
        t_window = sig["time_470"] - r_time
        mask = (t_window >= -window_sec) & (t_window <= window_sec)
        
        if not mask.any():
            continue
        
        # Plot aligned signal
        plt.plot(t_window[mask], sig["dF/F_filtered"][mask], color="black", lw=0.8)
        
        plt.axvline(0, color="gray", linestyle="--", alpha=0.8)
        plt.axvline(-5, color="gray", linestyle="--", alpha=0.8)
    
    plt.title(
        f"Animal {entry['animal_id']} | Genotype {entry['genotype']} | "
        f"Region {entry['region']} | Session {entry['session']} | Exp {entry['experiment']}"
    )
    plt.xlabel("Time relative to reward (s)")
    plt.ylabel(f"ΔF/F (high-pass >{highpass_cutoff} Hz)")
    plt.xlim(-window_sec, window_sec)
    plt.tight_layout()
    plt.show()


### Reward means

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.signal import butter, filtfilt
from scipy.interpolate import interp1d

# ----------- USER SETTINGS -----------
highpass_cutoff = 0.03  # Hz, high-pass filter cutoff
sampling_rate = 400     # Hz, adjust to your data
window_sec = 10         # seconds before and after reward
resample_points = 800   # points for uniform alignment (adjust if needed)

# ----------- FILTER FUNCTION -----------
def highpass_filter(signal, cutoff, fs, order=3):
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    b, a = butter(order, normal_cutoff, btype='high', analog=False)
    return filtfilt(b, a, signal)

# ----------- PERI-REWARD PLOTS (MEAN + SEM, no extrapolation) -----------
for entry in all_plots_baseline_sorted:
    sig = entry["signals"].copy()
    
    # Apply high-pass filter
    sig["dF/F_filtered"] = highpass_filter(sig["dF/F_baseline"], highpass_cutoff, sampling_rate)
    
    plt.figure(figsize=(14, 5))
    
    aligned_traces = []
    
    # Common resampled time vector
    t_resampled = np.linspace(-window_sec, window_sec, resample_points)

    for r_time in entry["reward_times"]:
        # Align time so reward = 0
        t_window = sig["time_470"] - r_time
        mask = (t_window >= -window_sec) & (t_window <= window_sec)
        if not np.any(mask):
            continue

        # Interpolate only within real data; do NOT extrapolate
        interp_func = interp1d(
            t_window[mask],
            sig["dF/F_filtered"][mask],
            kind='linear',
            bounds_error=False,
            fill_value=np.nan
        )
        aligned_traces.append(interp_func(t_resampled))

    if aligned_traces:
        aligned_array = np.vstack(aligned_traces)
        # Mean and SEM ignoring NaNs
        mean_trace = np.nanmean(aligned_array, axis=0)
        sem_trace = np.nanstd(aligned_array, axis=0) / np.sqrt(np.sum(~np.isnan(aligned_array), axis=0))

        # Plot mean + SEM
        plt.plot(t_resampled, mean_trace, color="red", lw=2, label="Mean ΔF/F")
        plt.fill_between(t_resampled, mean_trace - sem_trace, mean_trace + sem_trace,
                         color="red", alpha=0.3, label="SEM")

        # Add vertical lines
        plt.axvline(0, color="gray", linestyle="--", alpha=0.8)   # reward
        plt.axvline(-5, color="gray", linestyle="--", alpha=0.8)  # marker at -5 s

    plt.title(
        f"Animal {entry['animal_id']} | Genotype {entry['genotype']} | "
        f"Region {entry['region']} | Session {entry['session']} | Exp {entry['experiment']}"
    )
    plt.xlabel("Time relative to reward (s)")
    plt.ylabel(f"ΔF/F (high-pass >{highpass_cutoff} Hz)")
    plt.xlim(-window_sec, window_sec)
    plt.legend()
    plt.tight_layout()
    plt.show()

### Reward compiled means

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.interpolate import interp1d
from scipy.signal import butter, filtfilt
from collections import defaultdict

# ----------- USER SETTINGS -----------
highpass_cutoff = 0.03  # Hz
sampling_rate = 400     # Hz
window_sec = 10         # seconds before and after reward
resample_points = 800   # points for uniform alignment

# ----------- FILTER FUNCTION -----------
def highpass_filter(signal, cutoff, fs, order=3):
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    b, a = butter(order, normal_cutoff, btype='high', analog=False)
    return filtfilt(b, a, signal)

# ----------- GROUP DATA BY REGION + SESSION + GENOTYPE -----------
# ----------- GROUP DATA BY REGION + SESSION + GENOTYPE -----------
grouped_data = defaultdict(list)

for entry in all_plots_baseline_sorted:
    sig = entry["signals"].copy()
    sig["dF/F_filtered"] = highpass_filter(sig["dF/F_baseline"], highpass_cutoff, sampling_rate)

    t_resampled = np.linspace(-window_sec, window_sec, resample_points)
    aligned_traces = []

    for r_time in entry["reward_times"]:
        t_window = sig["time_470"] - r_time
        mask = (t_window >= -window_sec) & (t_window <= window_sec)
        if not np.any(mask):
            continue

        # Interpolate only within actual trace
        interp_func = np.interp(
            t_resampled,
            t_window[mask],
            sig["dF/F_filtered"][mask],
            left=np.nan,
            right=np.nan
        )
        aligned_traces.append(interp_func)

    if aligned_traces:
        aligned_array = np.vstack(aligned_traces)
        mean_trace = np.nanmean(aligned_array, axis=0)
        sem_trace = np.nanstd(aligned_array, axis=0) / np.sqrt(np.sum(~np.isnan(aligned_array), axis=0))

        key = (entry["region"], entry["session"], entry["genotype"])
        grouped_data[key].append({
            "animal_id": entry["animal_id"],
            "mean_trace": mean_trace,
            "sem_trace": sem_trace
        })

# ----------- PLOT GROUPED MEAN TRACES -----------
# ----------- PLOT GROUPED MEAN TRACES -----------
for (region, session, genotype), animal_traces in grouped_data.items():
    plt.figure(figsize=(14, 5))
    t_resampled = np.linspace(-window_sec, window_sec, resample_points)

    # Plot each animal
    for animal in animal_traces:
        plt.plot(t_resampled, animal["mean_trace"], lw=2, label=f"Animal {animal['animal_id']}")

    # Group mean ± SEM
    all_means = np.array([a["mean_trace"] for a in animal_traces])
    group_mean = np.nanmean(all_means, axis=0)
    group_sem = np.nanstd(all_means, axis=0) / np.sqrt(len(animal_traces))
    
    plt.plot(t_resampled, group_mean, color="black", lw=3, label="Group mean")
    plt.fill_between(t_resampled, group_mean - group_sem, group_mean + group_sem,
                     color="gray", alpha=0.3, label="Group SEM")

    # Vertical lines
    plt.axvline(0, color="gray", linestyle="--", alpha=0.8)
    plt.axvline(-5, color="gray", linestyle="--", alpha=0.8)

    plt.title(f"Region {region} | Session {session} | Genotype {genotype}")
    plt.xlabel("Time relative to reward (s)")
    plt.ylabel(f"ΔF/F (high-pass >{highpass_cutoff} Hz)")
    plt.xlim(-window_sec, window_sec)
    plt.legend()
    plt.tight_layout()
    plt.show()

### Reward collect

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.signal import butter, filtfilt

# ----------- USER SETTINGS -----------
highpass_cutoff = 0.03  # Hz
sampling_rate = 400     # Hz
window_sec = 10         # seconds before and after reward collect

# ----------- FILTER FUNCTION -----------
def highpass_filter(signal, cutoff, fs, order=3):
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    b, a = butter(order, normal_cutoff, btype='high', analog=False)
    return filtfilt(b, a, signal)

# ----------- PERI-REWARD-COLLECT PLOTS (0 = reward collect) -----------
for entry in all_plots_baseline_sorted:
    sig = entry["signals"].copy()
    
    # Apply high-pass filter
    sig["dF/F_filtered"] = highpass_filter(sig["dF/F_baseline"], highpass_cutoff, sampling_rate)
    
    plt.figure(figsize=(14, 5))
    
    for collect_time in entry["collect_times"]:
        if collect_time is None:
            continue
        
        # Align time so reward collect = 0
        t_window = sig["time_470"] - collect_time
        mask = (t_window >= -window_sec) & (t_window <= window_sec)
        
        if not mask.any():
            continue
        
        # Plot aligned signal
        plt.plot(t_window[mask], sig["dF/F_filtered"][mask], color="black", lw=0.8)
        
        # Reward collect marker at 0
        plt.axvline(0, color="gray", linestyle="--", alpha=0.8)
    
    plt.title(
        f"Animal {entry['animal_id']} | Genotype {entry['genotype']} | "
        f"Region {entry['region']} | Session {entry['session']} | Exp {entry['experiment']}"
    )
    plt.xlabel("Time relative to reward collect (s)")
    plt.ylabel(f"ΔF/F (high-pass >{highpass_cutoff} Hz)")
    plt.xlim(-window_sec, window_sec)
    plt.tight_layout()
    plt.show()

# Quantifying behavior

In [ ]:
# --------------------
# BEHAVIOR QUANTIFICATION
# --------------------

import numpy as np
import pandas as pd

# --------------------
# Helper functions
# --------------------

def compute_early_and_iti_time(events_df, early_window=10):
    """
    Compute total early and ITI time windows.
    
    early_window: seconds to count as early (includes tone + 5s post-reward)
    """
    reward_times = events_df.loc[
        events_df["Item_Name"] == "reward", "Evnt_Time"
    ].values

    n_rewards = len(reward_times)

    if n_rewards == 0:
        return 0.0, np.nan  # habituation or no rewards

    session_start = events_df["Evnt_Time"].min()
    session_end = events_df["Evnt_Time"].max()
    total_time = session_end - session_start

    early_time = n_rewards * early_window
    iti_time = max(total_time - early_time, 0)

    return early_time, iti_time


def compute_reward_collect_delays(events_df):
    """
    Compute mean reward collection delay.
    """
    reward_times = events_df[events_df["Item_Name"] == "reward"]["Evnt_Time"].values
    iti_times = events_df[events_df["Item_Name"].str.contains("ITI_head_entry")]["Evnt_Time"].values
    early_times = events_df[events_df["Item_Name"].str.contains("early_head_entry")]["Evnt_Time"].values
    tone_times = events_df[events_df["Item_Name"] == "Sound_On #1"]["Evnt_Time"].values

    # Make sure the first tone at 0 is included
    if len(tone_times) == 0 or tone_times[0] != 0:
        tone_times = np.insert(tone_times, 0, 0)  # prepend 0

    reward_delays = []

    for i, r_time in enumerate(reward_times):
        next_reward_time = reward_times[i + 1] if i + 1 < len(reward_times) else np.inf

        candidate_iti = iti_times[(iti_times > r_time) & (iti_times < next_reward_time)]
        if len(candidate_iti) > 0:
            reward_delays.append(candidate_iti[0] - r_time)
            continue

        candidate_early = early_times[(early_times > r_time) & (early_times < next_reward_time)]
        if len(candidate_early) > 0:
            reward_delays.append(candidate_early[0] - r_time)
            continue

    return float(np.mean(reward_delays)) if reward_delays else np.nan


def quantify_behavior(events_df):
    """
    Compute head entry metrics and rates.
    """
    early_entries = events_df[events_df["Item_Name"].str.contains("early_head_entry")]
    iti_entries = events_df[events_df["Item_Name"].str.contains("ITI_head_entry")]

    n_early = early_entries["Arg1_Value"].iloc[-1] if not early_entries.empty else 0
    n_iti_all = iti_entries["Arg1_Value"].iloc[-1] if not iti_entries.empty else 0

    reward_times = events_df[events_df["Item_Name"] == "reward"]["Evnt_Time"].values
    iti_times = iti_entries["Evnt_Time"].values if not iti_entries.empty else np.array([])

    # ITI head entries within 5 s after reward
    iti_within5_mask = np.zeros(len(iti_times), dtype=bool)
    for i, iti_time in enumerate(iti_times):
        iti_within5_mask[i] = np.any(
            (iti_time > reward_times) & (iti_time <= reward_times + 5)
        )

    n_iti_within5 = int(np.sum(iti_within5_mask))
    n_iti_excluding_within5 = n_iti_all - n_iti_within5

    n_early_including_ITI_within5 = n_early + n_iti_within5

    # --- TIME WINDOWS ---
    early_time, iti_time = compute_early_and_iti_time(events_df)
    session_start = events_df["Evnt_Time"].min()
    session_end = events_df["Evnt_Time"].max()
    total_session_duration = session_end - session_start

    # --- RATES (entries / second)
    early_rate = n_early_including_ITI_within5 / early_time if early_time > 0 else np.nan
    iti_rate = n_iti_excluding_within5 / iti_time if iti_time > 0 else np.nan
    early_to_iti_rate_fraction = early_rate / iti_rate if iti_rate > 0 else np.nan

    # --- TOTAL HEAD ENTRY RATE (entire session) ---
    total_head_entries = n_early + n_iti_all
    total_head_entry_rate = total_head_entries / total_session_duration if total_session_duration > 0 else np.nan

    # --- EXCLUSIVE RATES (5 s after reward removed from everything) ---
    n_rewards = len(reward_times)
    iti_time_exclusive = max(iti_time - n_rewards * 5, 0) if not np.isnan(iti_time) else np.nan

    early_rate_exclusive = n_early / early_time if early_time > 0 else np.nan
    iti_rate_exclusive = n_iti_excluding_within5 / iti_time_exclusive if iti_time_exclusive > 0 else np.nan
    early_to_iti_rate_fraction_exclusive = (
        early_rate_exclusive / iti_rate_exclusive
        if iti_rate_exclusive > 0 and not np.isnan(early_rate_exclusive)
        else np.nan
    )

    reward_collect_delay_mean = compute_reward_collect_delays(events_df)

    return {
        # Counts
        "n_early_head_entries": n_early,
        "n_ITI_all": n_iti_all,
        "n_ITI_within5": n_iti_within5,
        "n_ITI_excluding_within5": n_iti_excluding_within5,
        "n_early_including_ITI_within5": n_early_including_ITI_within5,

        # Time (seconds)
        "early_time_s": early_time,
        "ITI_time_s": iti_time,
        "total_session_duration_s": total_session_duration,  # NEW

        # Rates (entries / second)
        "early_rate": early_rate,
        "ITI_rate": iti_rate,
        "early_to_iti_rate_fraction": early_to_iti_rate_fraction,
        "total_head_entry_rate": total_head_entry_rate,  # NEW

        # Exclusive (5s after reward removed from everything)
        "early_rate_exclusive": early_rate_exclusive,
        "ITI_rate_exclusive": iti_rate_exclusive,
        "ITI_time_exclusive_s": iti_time_exclusive,
        "early_to_ITI_rate_fraction_exclusive": early_to_iti_rate_fraction_exclusive,

        # Other metrics
        "reward_collect_delay_mean": reward_collect_delay_mean
    }


# --------------------
# MAIN LOOP: QUANTIFY BEHAVIOR AND SAVE CSV
# --------------------

behavior_summary = []

for exp_folder in input_path.iterdir():
    if not exp_folder.is_dir():
        continue

    print(f"\nProcessing experiment: {exp_folder.name}")

    for sub_name in session_order:
        sub_path = exp_folder / sub_name
        if not sub_path.exists():
            print(f"  ⚠ Missing session folder: {sub_name}")
            continue

        session_files = collect_session_files(sub_path)

        for beh_file in session_files["behavior"]:
            try:
                animal_id = get_animal_id_from_behavior_csv(beh_file)
                genotype = genotype_map.get(animal_id, "Unknown")
                events_df = extract_behavior_events(beh_file)  # raw Evnt_Time
                metrics = quantify_behavior(events_df)

                behavior_summary.append({
                    "session": sub_name,
                    "animal_id": animal_id,
                    "genotype": genotype,
                    **metrics
                })
                print(f"  ✅ Processed: {beh_file.name}")

            except Exception as e:
                print(f"  ❌ {beh_file.name}: {e}")

# Convert to DataFrame and save
if behavior_summary:
    behavior_summary_df = pd.DataFrame(behavior_summary)
    behavior_summary_df["session"] = pd.Categorical(
        behavior_summary_df["session"], categories=session_order, ordered=True
    )
    behavior_summary_df["genotype"] = pd.Categorical(
        behavior_summary_df["genotype"], categories=["WT", "KI"], ordered=True
    )
    behavior_summary_df = behavior_summary_df.sort_values(
        by=["session", "genotype", "animal_id"]
    )

    output_path = input_path.parent / "Pavlovian_summary.csv"
    behavior_summary_df.to_csv(output_path, index=False)
    print(f"\n✅ Behavior summary saved to: {output_path}")
else:
    print("\n⚠ No behavior data found, CSV not created.")

# Plotting behavior

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

dot_size = 10  # size of head entry dots

# --------------------
# Gather all behavior files with metadata
# --------------------
all_files = []

for exp_folder in input_path.iterdir():
    if not exp_folder.is_dir():
        continue
    metadata = parse_exp_folder(exp_folder.name)
    animal_A, animal_B = metadata["animal_A"], metadata["animal_B"]

    for sub_name in session_order:
        sub_path = exp_folder / sub_name
        if not sub_path.exists():
            continue
        session_files = collect_session_files(sub_path)

        for beh_file in session_files["behavior"]:
            try:
                animal_id = get_animal_id_from_behavior_csv(beh_file)
                genotype = genotype_map.get(animal_id, "Unknown")
                all_files.append({
                    "session": sub_name,
                    "genotype": genotype,
                    "animal_id": animal_id,
                    "file": beh_file
                })
            except Exception as e:
                print(f"  ❌ {beh_file.name}: {e}")

# --------------------
# Convert to DataFrame and sort
# --------------------
df_files = pd.DataFrame(all_files)
df_files["session"] = pd.Categorical(df_files["session"], categories=session_order, ordered=True)
df_files["genotype"] = pd.Categorical(df_files["genotype"], categories=["WT", "KI"], ordered=True)
df_files = df_files.sort_values(by=["session", "genotype", "animal_id"])

# --------------------
# Plot sorted files
# --------------------
for _, row in df_files.iterrows():
    try:
        events_df = extract_behavior_events(row["file"])

        plt.figure(figsize=(30, 2))
        plt.title(f"{row['animal_id']} ({row['genotype']}) - {row['session']}")
        plt.xlabel("Time (s)")
        plt.yticks([])

        # Head entries as dots
        for head_type, color in [("early_head_entry", "red"), ("ITI_head_entry", "blue")]:
            times = events_df[events_df["Item_Name"].str.contains(head_type)]["Evnt_Time"].values
            plt.scatter(times, [1]*len(times), color=color, s=dot_size, 
                        label=f"{head_type} (n={len(times)})")

        # Reward and tone as vertical lines with exact match counts
        for line_type, color in [("reward", "green"), ("Sound_On #1", "orange")]:
            times = events_df[events_df["Item_Name"] == line_type]["Evnt_Time"].values

            # Only for tone, and only if not Habituation, ensure first tone at 0 is included
            if line_type == "Sound_On #1" and row['session'] != "Habituation":
                if len(times) == 0 or times[0] != 0:
                    times = np.insert(times, 0, 0)

            n_events = len(times)
            for i, t in enumerate(times):
                label = f"{line_type} (n={n_events})" if i == 0 else ""
                plt.vlines(t, 0, 1.2, color=color, linewidth=1.5, label=label)

        plt.ylim(0, 1.3)
        plt.legend(loc="upper right")
        plt.tight_layout()
        plt.show()

    except Exception as e:
        print(f"  ❌ {row['file'].name}: {e}")